In [1]:
# ============================================================
# STEP2 と STEP3〜3.3 の保存済み結果を比較（コンソール出力）
#  - ルート: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No
#  - 読み込み:
#     STEP2_old_MDS_*/<dataset>/cluster_labels_*.csv
#     STEP3_new_MDS_*/<dataset>/ClusterAssign_*.csv（12本/データセット想定）
#     STEP3.2_signlessCorr_MDS_*/<dataset>/cluster_labels_*.csv（linear/nonlinear）
#     STEP3.3_olddata_MDS_*/<dataset>/ClusterAssign_*.csv（12本/データセット想定）
#  - 比較: 同一 id の共通列（top3/cumeig × 6指標）で ARI
#  - 出力: step2 vs new / step2 vs signless-linear / step2 vs olddata / new vs signless-linear / new vs olddata（各 dataset）
# ============================================================

suppressPackageStartupMessages({
  if (!requireNamespace("mclust", quietly=TRUE)) install.packages("mclust")
  library(mclust)  # adjustedRandIndex
  if (!requireNamespace("dplyr", quietly=TRUE))  install.packages("dplyr")
  library(dplyr)
  if (!requireNamespace("stringr", quietly=TRUE)) install.packages("stringr")
  library(stringr)
})

root_base <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
datasets  <- c("OH","FP")
indices6  <- c("ptbiserial","db","ch","gap","dunn","silhouette")
modes2    <- c("top3","cumeig")

# ---------- ユーティリティ ----------
latest_dir <- function(base, pattern){
  cand <- list.dirs(base, full.names = TRUE, recursive = FALSE)
  cand <- cand[grepl(pattern, basename(cand))]
  if (length(cand) == 0) return(NA_character_)
  cand[order(file.info(cand)$mtime, decreasing = TRUE)][1]
}

latest_file <- function(dir_path, pattern){
  cand <- list.files(dir_path, pattern = pattern, full.names = TRUE)
  if (length(cand) == 0) return(NA_character_)
  cand[order(file.info(cand)$mtime, decreasing = TRUE)][1]
}

safe_read_csv <- function(path){
  if (is.na(path)) return(NULL)
  tryCatch(utils::read.csv(path, check.names = FALSE), error = function(e) NULL)
}

# 追加：partition の整列ヘルパー
align_partition <- function(cl, X){
  rn <- rownames(X)
  if (is.null(cl)) {
    out <- rep(NA_integer_, length(rn)); names(out) <- rn; return(out)
  }
  cl <- as.integer(cl)

  # 1) 名前が無ければ行名をそのまま付与（行順と同じとみなす）
  if (is.null(names(cl))) {
    if (length(cl) == length(rn)) {
      names(cl) <- rn
    } else {
      out <- rep(NA_integer_, length(rn)); names(out) <- rn; return(out)
    }
  }

  # 2) 名前があっても rn を全てカバーしていなければ、長さ一致なら行順で上書き
  if (!all(rn %in% names(cl))) {
    if (length(cl) == length(rn)) {
      names(cl) <- rn
    } else {
      out <- rep(NA_integer_, length(rn)); names(out) <- rn; return(out)
    }
  }

  cl[rn]
}


# --- ClusterAssign_*（1ファイル=1列のラベル想定）を束ねて wide へ ---
read_step3_assign_bundle <- function(dir_ds){
  if (is.na(dir_ds) || !dir.exists(dir_ds)) return(NULL)
  files <- list.files(dir_ds, pattern = "^ClusterAssign_(top3|cumeig)_(ptbiserial|db|ch|gap|dunn|silhouette)_.*\\.csv$",
                      full.names = TRUE)
  if (!length(files)) return(NULL)

  out <- NULL
  for (fp in files){
    m <- str_match(basename(fp), "^ClusterAssign_(top3|cumeig)_([a-z]+)_.*\\.csv$")
    mode <- m[2]; idx <- m[3]
    df <- safe_read_csv(fp); if (is.null(df)) next

    # id 列（優先: "id"、無ければ1列目）
    id_col <- if ("id" %in% names(df)) "id" else names(df)[1]
    ids <- df[[id_col]]

    # クラスタ列（数値/整数/因子の列を優先。候補が複数なら最後）
    num_mask <- vapply(df, function(v) is.numeric(v) || is.integer(v) || is.factor(v), logical(1))
    num_cols <- names(df)[num_mask]
    if (length(num_cols) == 0) next
    cl <- df[[tail(num_cols, 1)]]
    cl <- as.integer(as.factor(cl))

    colname <- paste(mode, idx, sep = "_")
    tmp <- data.frame(id = ids, tmp = cl, check.names = FALSE)
    names(tmp)[2] <- colname

    out <- if (is.null(out)) tmp else merge(out, tmp, by="id", all=TRUE)
  }
  out
}

# --- signless の cluster_labels_* を読み、linear_* を標準名（top3_* / cumeig_*）に変換 ---
read_signless_linear_labels <- function(dir_ds){
  if (is.na(dir_ds) || !dir.exists(dir_ds)) return(NULL)
  fp <- latest_file(dir_ds, "^cluster_labels_.*\\.csv$")
  df <- safe_read_csv(fp); if (is.null(df)) return(NULL)
  if (!"id" %in% names(df)) return(NULL)

  cols <- setdiff(names(df), "id")
  keep <- grepl("^linear_", cols)
  cols_keep <- cols[keep]
  if (!length(cols_keep)) return(data.frame(id=df$id))
  std_names <- sub("^linear_", "", cols_keep)

  out <- df["id"]
  for (i in seq_along(cols_keep)){
    out[[std_names[i]]] <- as.integer(as.factor(df[[cols_keep[i]]]))
  }
  out
}

# --- STEP2 の cluster_labels_* をそのまま読む（既に top3_* / cumeig_* の想定） ---
read_step2_labels <- function(dir_ds){
  if (is.na(dir_ds) || !dir.exists(dir_ds)) return(NULL)
  fp <- latest_file(dir_ds, "cluster_labels_.*\\.csv$")
  df <- safe_read_csv(fp); if (is.null(df)) return(NULL)
  if (!"id" %in% names(df)) {
    # 先頭列が id の場合
    names(df)[1] <- "id"
  }
  df$id <- df$id
  for (nm in setdiff(names(df), "id")){
    df[[nm]] <- as.integer(as.factor(df[[nm]]))
  }
  df
}

# --- 共通列だけ ARI を計算 ---
pairwise_ari <- function(A, B, label=""){
  if (is.null(A) || is.null(B)) {
    cat(sprintf("[%s] (missing)\n", label)); return(invisible(NULL))
  }
  colsA <- intersect(setdiff(names(A), "id"), paste(rep(modes2, each=length(indices6)), indices6, sep="_"))
  colsB <- intersect(setdiff(names(B), "id"), paste(rep(modes2, each=length(indices6)), indices6, sep="_"))
  common <- intersect(colsA, colsB)
  if (!length(common)) {
    cat(sprintf("[%s] no common columns.\n", label)); return(invisible(NULL))
  }
  M <- merge(A[, c("id", common)], B[, c("id", common)], by="id", suffixes = c(".A",".B"))
  rows <- nrow(M)
  out <- lapply(common, function(cn){
    a <- M[[paste0(cn,".A")]]; b <- M[[paste0(cn,".B")]]
    ok <- !(is.na(a) | is.na(b))
    n_ok <- sum(ok)
    ari <- if (n_ok >= 2) mclust::adjustedRandIndex(as.factor(a[ok]), as.factor(b[ok])) else NA_real_
    data.frame(condition = cn, n = n_ok, ARI = ari, stringsAsFactors = FALSE)
  })
  tbl <- bind_rows(out)
  tbl <- tbl[order(tbl$condition), ]
  cat(sprintf("\n-- %s --\n", label))
  print(tbl, row.names = FALSE)
  invisible(tbl)
}

# ---------- 対象ディレクトリを自動検出 ----------
dir_step2      <- latest_dir(root_base, "^STEP2_")
dir_step3_new  <- latest_dir(root_base, "^STEP3_new")
dir_signless   <- latest_dir(root_base, "^STEP3\\.2_signlessCorr")
dir_step3_old  <- latest_dir(root_base, "^STEP3\\.3_olddata")

cat("Resolved root: ", root_base, "\n", sep="")
cat("STEP2:      ", dir_step2,     "\n", sep="")
cat("STEP3 new:  ", dir_step3_new, "\n", sep="")
cat("signless:   ", dir_signless,  "\n", sep="")
cat("STEP3 old:  ", dir_step3_old, "\n", sep="")

# ---------- 読み込み & 比較 ----------
for (ds in datasets){
  cat("\n================ DATASET: ", ds, " ================\n", sep="")

  # STEP2
  step2_ds  <- if (!is.na(dir_step2))  file.path(dir_step2,  ds) else NA_character_
  L_step2   <- read_step2_labels(step2_ds)

  # STEP3 new: ClusterAssign_* 12本を束ねる
  new_ds    <- if (!is.na(dir_step3_new)) file.path(dir_step3_new, ds) else NA_character_
  L_new     <- read_step3_assign_bundle(new_ds)

  # signless: cluster_labels_* の linear_* だけを標準名へ
  sgn_ds    <- if (!is.na(dir_signless)) file.path(dir_signless, ds) else NA_character_
  L_sgn_lin <- read_signless_linear_labels(sgn_ds)

  # STEP3 olddata: ClusterAssign_* 12本を束ねる
  old_ds    <- if (!is.na(dir_step3_old)) file.path(dir_step3_old, ds) else NA_character_
  L_old     <- read_step3_assign_bundle(old_ds)

  # どの列があるかを簡単に表示
  avail_cols <- function(x) paste(intersect(setdiff(names(x), "id"),
                                            paste(rep(modes2, each=length(indices6)), indices6, sep="_")), collapse=", ")
  cat("[available columns]\n")
  cat(" step2:      ", if(!is.null(L_step2)) avail_cols(L_step2) else "(none)", "\n", sep="")
  cat(" new:        ", if(!is.null(L_new))   avail_cols(L_new)   else "(none)", "\n", sep="")
  cat(" signless L: ", if(!is.null(L_sgn_lin)) avail_cols(L_sgn_lin) else "(none)", "\n", sep="")
  cat(" olddata:    ", if(!is.null(L_old))   avail_cols(L_old)   else "(none)", "\n", sep="")

  # ---- ペア比較（共通列だけ ARI）----
  pairwise_ari(L_step2, L_new,      "STEP2 vs NEW (linear)")
  pairwise_ari(L_step2, L_sgn_lin,  "STEP2 vs SIGNLESS (linear)")
  pairwise_ari(L_step2, L_old,      "STEP2 vs OLDDATA (linear)")
  pairwise_ari(L_new,   L_sgn_lin,  "NEW vs SIGNLESS (linear)")
  pairwise_ari(L_new,   L_old,      "NEW vs OLDDATA (linear)")
}
cat("\nDONE: All comparisons printed to console.\n")


Resolved root: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No
STEP2:      /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP2_old_MDS_20250904
STEP3 new:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904
signless:   /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3.2_signlessCorr_MDS_20250904
STEP3 old:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3.3_olddata_MDS_20250904

================ DATASET: OH ================
[available columns]
 step2:      (none)
 new:        cumeig_ch, cumeig_db, cumeig_dunn, cumeig_gap, cumeig_ptbiserial, cumeig_silhouette, top3_ch, top3_db, top3_dunn, top3_gap, top3_ptbiserial, top3_silhouette
 signless L: top3_silhouette, top3_dunn, top3_gap, top3_ch, top3_db, top3_ptbiserial, cumeig_silhouette, cumeig_dunn, cumeig_gap, cumeig_ch, cumeig_db, cumeig_ptbiserial
 olddata:    cumeig_ch, cumeig_db, cumeig_dunn, cumeig_gap, cumeig_ptbiserial, cumeig_silhouette, top

In [6]:
# ============================================================
# AXIS-PAIR ARI 一括計算スクリプト（console 出力）
# - 既存の MDScoords_* と NbClust を使い、軸集合（D1-D2/D1-D3/D2-D3/Top3/K80）で
#   それぞれクラスタ分割し、軸集合ペア間のARIを指標ごとに出力
# - "gap/ch/db/ptbiserial" も data=座標 で計算（diss不要）
# - 以前のバグ（MDScoordsの拾い間違い）を回避
# ============================================================

suppressPackageStartupMessages({
  if (!require(mclust))   install.packages("mclust");   library(mclust)   # ARI
  if (!require(NbClust))  install.packages("NbClust");  library(NbClust)  # 指標
  if (!require(dplyr))    install.packages("dplyr");    library(dplyr)
  if (!require(parallel)) install.packages("parallel"); library(parallel)
})

# ---- コンソール出力（UTF-8安全） ----
invisible(try(Sys.setlocale("LC_CTYPE", "ja_JP.UTF-8"), silent = TRUE))
say <- function(..., sep="") {
  txt <- paste0(..., collapse = sep)
  cat(enc2native(txt)); if (!grepl("\n$", txt)) cat("\n")
}

# ---- 設定（必要に応じて変更） ----
root_base    <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
datasets     <- c("OH","FP")
steps_to_run <- c("new","signless","olddata")   # 実行したいSTEP
modes_for_step <- list(                         # stepごとの mode
  new      = c("linear"),
  olddata  = c("linear"),
  signless = c("linear","nonlinear")
)
indices <- c("silhouette","dunn","gap","ch","db","ptbiserial")  # 指標
N_CORES <- max(1, min(4, parallel::detectCores(logical = TRUE))) # 使いすぎ防止

# ---- ユーティリティ ----
latest_file <- function(dir_path, pattern){
  cand <- list.files(dir_path, pattern = pattern, full.names = TRUE)
  if (!length(cand)) return(NA_character_)
  info <- file.info(cand)
  cand[order(info$mtime, decreasing = TRUE)][1]
}

find_latest_stepdir <- function(root, step_regex){
  dirs <- list.dirs(root, full.names = TRUE, recursive = FALSE)
  if (!length(dirs)) return(NA_character_)
  hit  <- dirs[grepl(step_regex, basename(dirs))]
  if (!length(hit)) return(NA_character_)
  hit[order(basename(hit))][length(hit)]
}

# MDScoords 読み込み（dim_mode = "cumeig" or "top3"）
load_mds_coords <- function(root, step, dataset, mode, dim_mode){
  step_dir <- switch(step,
    "new"      = find_latest_stepdir(root, "^STEP3_new"),
    "olddata"  = find_latest_stepdir(root, "^STEP3\\.3_olddata"),
    "signless" = find_latest_stepdir(root, "^STEP3\\.2_signlessCorr"),
    stop(sprintf("Unknown step: %s", step), call. = FALSE)
  )
  if (is.na(step_dir) || !dir.exists(step_dir)) {
    stop(sprintf("Step dir not found for '%s' under %s", step, root), call. = FALSE)
  }
  ds_dir <- file.path(step_dir, dataset)
  if (!dir.exists(ds_dir)) {
    stop(sprintf("Dataset dir not found: %s", ds_dir), call. = FALSE)
  }

  patt <- if (step == "signless") {
    # 例: MDScoords_linear_cumeig_OH_*.csv / MDScoords_nonlinear_top3_FP_*.csv
    sprintf("^MDScoords_%s_%s_%s_.*\\.csv$", mode, dim_mode, dataset)
  } else {
    # 例: MDScoords_cumeig_OH_*.csv / MDScoords_top3_FP_*.csv
    sprintf("^MDScoords_%s_%s_.*\\.csv$", dim_mode, dataset)
  }

  fp <- latest_file(ds_dir, patt)
  if (is.na(fp)) {
    stop(sprintf("MDScoords not found in: %s (pattern=%s)", ds_dir, patt), call. = FALSE)
  }

  df <- utils::read.csv(fp, check.names = FALSE, stringsAsFactors = FALSE)
  id_col <- if ("id" %in% names(df)) "id" else names(df)[1]
  ids <- df[[id_col]]
  num_mask <- vapply(df, is.numeric, logical(1))
  X <- as.matrix(df[, num_mask, drop = FALSE])
  rownames(X) <- ids

  if (ncol(X) < 2) stop(sprintf("Numeric columns < 2 in %s", fp), call. = FALSE)

  say(sprintf("[MDS file] %s", fp))
  list(dir = ds_dir, file = fp, X = X)
}

# NbClust を安全に（Ward.D2、euclid、k探索）
safe_nbclust <- function(X, index_name){
  # k探索幅（広すぎると遅い）：データ規模で控えめに
  maxnc <- min(30, nrow(X) - 1)  # 必要なら 100 などに変更
  res <- tryCatch({
    NbClust(data = X,
            diss = NULL,
            distance = "euclidean",
            min.nc = 2,
            max.nc = maxnc,
            method = "ward.D2",
            index = index_name)
  }, error = function(e) e)
  res
}

# 軸集合ごとにクラスタ分割（全 index 分）を作る
compute_labels_for_axes <- function(axis_mats, indices){
  # axis_mats: named list of matrices (e.g., D1-D2, D1-D3, D2-D3, Top3, K80)
  out <- list()
  for (idx in indices){
    labs <- lapply(axis_mats, function(X){
      res <- safe_nbclust(X, idx)
      if (inherits(res, "error")) {
        warning(sprintf("NbClust failed (index=%s): %s", idx, res$message))
        return(NA_integer_)
      }
    cl <- align_partition(res$Best.partition, X)
    })
    out[[idx]] <- labs
  }
  out
}

# 軸集合ペアの ARI テーブル
pairs_ARI_table <- function(labels_by_axis){
  keys <- names(labels_by_axis)
  prs <- t(combn(keys, 2))
  rows <- apply(prs, 1, function(p){
    a <- labels_by_axis[[p[1]]]
    b <- labels_by_axis[[p[2]]]
    ok <- !(is.na(a) | is.na(b))
    n_ok <- sum(ok)
    ari <- if (n_ok >= 2) mclust::adjustedRandIndex(as.factor(a[ok]), as.factor(b[ok])) else NA_real_
    c(from = p[1], to = p[2], n = n_ok, ARI = ari)
  })
  as.data.frame(t(rows), stringsAsFactors = FALSE) |>
    dplyr::mutate(n = as.integer(n), ARI = as.numeric(ARI))
}

# ---- メイン処理 ----
say(sprintf("Resolved root: %s", root_base))

for (step in steps_to_run){
  for (dataset in datasets){
    for (mode in modes_for_step[[step]]){

      say("")
      say(sprintf("==== AXIS-PAIR ARI (step=%s, dataset=%s, mode=%s) ====", step, dataset, mode))

      # MDS 座標読み込み
      obj_cum  <- load_mds_coords(root_base, step, dataset, mode, "cumeig")
      obj_top3 <- load_mds_coords(root_base, step, dataset, mode, "top3")

      Xc <- obj_cum$X
      Xt <- obj_top3$X

      # 軸の存在チェック
      if (ncol(Xc) < 3 && ncol(Xt) < 3) {
        stop("Both cumeig/top3 have <3 numeric columns. Cannot build D1-D3/D2-D3/Top3.", call. = FALSE)
      }

      # 軸集合（K80は cumeig 全列、Top3 は top3ファイルがあればそれ、無ければ Xc[,1:3]）
      axis_mats <- list()
      axis_mats[["K80"]]   <- Xc
      axis_mats[["Top3"]]  <- if (ncol(Xt) >= 3) Xt[, 1:3, drop = FALSE] else Xc[, 1:3, drop = FALSE]
      axis_mats[["D1-D2"]] <- Xc[, 1:2, drop = FALSE]
      if (ncol(Xc) >= 3) {
        axis_mats[["D1-D3"]] <- Xc[, c(1,3), drop = FALSE]
        axis_mats[["D2-D3"]] <- Xc[, c(2,3), drop = FALSE]
      } else {
        # cumeig に3軸ない場合は top3 から拝借
        axis_mats[["D1-D3"]] <- Xt[, c(1,3), drop = FALSE]
        axis_mats[["D2-D3"]] <- Xt[, c(2,3), drop = FALSE]
      }

      say(sprintf("K80 dimensions: %d", ncol(axis_mats[["K80"]])))

      # ---- クラスタ割当を並列計算（index × 軸集合） ----
      # 軸ごとに独立なので indexごとに順次、内部で軸を並列化しても良いが、
      # NbClustはスレッド重いので控えめに。
      labels_all <- list()
      for (idx in indices){
        # 軸ごとに並列
        labs <- mclapply(names(axis_mats), function(axn){
          X <- axis_mats[[axn]]
          res <- safe_nbclust(X, idx)
          if (inherits(res, "error")) {
            warning(sprintf("NbClust failed (index=%s, axis=%s): %s", idx, axn, res$message))
            return(NA_integer_)
          }
        cl <- align_partition(res$Best.partition, X)
        }, mc.cores = N_CORES)
        names(labs) <- names(axis_mats)
        labels_all[[idx]] <- labs
      }

      # ---- indexごとに ARI 表を出力 ----
      for (idx in indices){
        say(sprintf("  -- index=%s --", idx))
        tbl <- pairs_ARI_table(labels_all[[idx]])
        # 見やすさのため、from/to/ n / ARI（降順）で出す
        tbl <- tbl |>
          dplyr::mutate(ARI = round(ARI, 9)) |>
          dplyr::arrange(dplyr::desc(ARI))
        print(tbl, row.names = FALSE)
      }

    }
  }
}

say("\nDONE: All ARI tables printed to console.")


Warning message in Sys.setlocale("LC_CTYPE", "ja_JP.UTF-8"):
"OS reports request to set locale to "ja_JP.UTF-8" cannot be honored"


Resolved root: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No

==== AXIS-PAIR ARI (step=new, dataset=OH, mode=linear) ====
[MDS file] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/OH/MDScoords_cumeig_OH_20250904_153003.csv
[MDS file] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/OH/MDScoords_top3_OH_20250904_153003.csv
K80 dimensions: 151


Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"


  -- index=silhouette --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D3 1  NA
   K80 D2-D3 1  NA
  Top3 D1-D2 1  NA
  Top3 D1-D3 1  NA
  Top3 D2-D3 1  NA
 D1-D2 D1-D3 1  NA
 D1-D2 D2-D3 1  NA
 D1-D3 D2-D3 1  NA
  -- index=dunn --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D3 1  NA
   K80 D2-D3 1  NA
  Top3 D1-D2 1  NA
  Top3 D1-D3 1  NA
  Top3 D2-D3 1  NA
 D1-D2 D1-D3 1  NA
 D1-D2 D2-D3 1  NA
 D1-D3 D2-D3 1  NA
  -- index=gap --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D3 1  NA
   K80 D2-D3 1  NA
  Top3 D1-D2 1  NA
  Top3 D1-D3 1  NA
  Top3 D2-D3 1  NA
 D1-D2 D1-D3 1  NA
 D1-D2 D2-D3 1  NA
 D1-D3 D2-D3 1  NA
  -- index=ch --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D3 1  NA
   K80 D2-D3 1  NA
  Top3 D1-D2 1  NA
  Top3 D1-D3 1  NA
  Top3 D2-D3 1  NA
 D1-D2 D1-D3 1  NA
 D1-D2 D2-D3 1  NA
 D1-D3 D2-D3 1  NA
  -- index=db --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D

Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"


...

  -- index=silhouette --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D3 1  NA
   K80 D2-D3 1  NA
  Top3 D1-D2 1  NA
  Top3 D1-D3 1  NA
  Top3 D2-D3 1  NA
 D1-D2 D1-D3 1  NA
 D1-D2 D2-D3 1  NA
 D1-D3 D2-D3 1  NA
  -- index=dunn --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D3 1  NA
   K80 D2-D3 1  NA
  Top3 D1-D2 1  NA
  Top3 D1-D3 1  NA
  Top3 D2-D3 1  NA
 D1-D2 D1-D3 1  NA
 D1-D2 D2-D3 1  NA
 D1-D3 D2-D3 1  NA
  -- index=gap --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D3 1  NA
   K80 D2-D3 1  NA
  Top3 D1-D2 1  NA
  Top3 D1-D3 1  NA
  Top3 D2-D3 1  NA
 D1-D2 D1-D3 1  NA
 D1-D2 D2-D3 1  NA
 D1-D3 D2-D3 1  NA
  -- index=ch --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D3 1  NA
   K80 D2-D3 1  NA
  Top3 D1-D2 1  NA
  Top3 D1-D3 1  NA
  Top3 D2-D3 1  NA
 D1-D2 D1-D3 1  NA
 D1-D2 D2-D3 1  NA
 D1-D3 D2-D3 1  NA
  -- index=db --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D

Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"


  -- index=silhouette --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D3 1  NA
   K80 D2-D3 1  NA
  Top3 D1-D2 1  NA
  Top3 D1-D3 1  NA
  Top3 D2-D3 1  NA
 D1-D2 D1-D3 1  NA
 D1-D2 D2-D3 1  NA
 D1-D3 D2-D3 1  NA
  -- index=dunn --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D3 1  NA
   K80 D2-D3 1  NA
  Top3 D1-D2 1  NA
  Top3 D1-D3 1  NA
  Top3 D2-D3 1  NA
 D1-D2 D1-D3 1  NA
 D1-D2 D2-D3 1  NA
 D1-D3 D2-D3 1  NA
  -- index=gap --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D3 1  NA
   K80 D2-D3 1  NA
  Top3 D1-D2 1  NA
  Top3 D1-D3 1  NA
  Top3 D2-D3 1  NA
 D1-D2 D1-D3 1  NA
 D1-D2 D2-D3 1  NA
 D1-D3 D2-D3 1  NA
  -- index=ch --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D3 1  NA
   K80 D2-D3 1  NA
  Top3 D1-D2 1  NA
  Top3 D1-D3 1  NA
  Top3 D2-D3 1  NA
 D1-D2 D1-D3 1  NA
 D1-D2 D2-D3 1  NA
 D1-D3 D2-D3 1  NA
  -- index=db --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D

Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"
Warning message in mclapply(names(axis_mats), function(axn) {:
"all scheduled cores encountered errors in user code"


  -- index=silhouette --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D3 1  NA
   K80 D2-D3 1  NA
  Top3 D1-D2 1  NA
  Top3 D1-D3 1  NA
  Top3 D2-D3 1  NA
 D1-D2 D1-D3 1  NA
 D1-D2 D2-D3 1  NA
 D1-D3 D2-D3 1  NA
  -- index=dunn --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D3 1  NA
   K80 D2-D3 1  NA
  Top3 D1-D2 1  NA
  Top3 D1-D3 1  NA
  Top3 D2-D3 1  NA
 D1-D2 D1-D3 1  NA
 D1-D2 D2-D3 1  NA
 D1-D3 D2-D3 1  NA
  -- index=gap --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D3 1  NA
   K80 D2-D3 1  NA
  Top3 D1-D2 1  NA
  Top3 D1-D3 1  NA
  Top3 D2-D3 1  NA
 D1-D2 D1-D3 1  NA
 D1-D2 D2-D3 1  NA
 D1-D3 D2-D3 1  NA
  -- index=ch --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D3 1  NA
   K80 D2-D3 1  NA
  Top3 D1-D2 1  NA
  Top3 D1-D3 1  NA
  Top3 D2-D3 1  NA
 D1-D2 D1-D3 1  NA
 D1-D2 D2-D3 1  NA
 D1-D3 D2-D3 1  NA
  -- index=db --
  from    to n ARI
   K80  Top3 1  NA
   K80 D1-D2 1  NA
   K80 D1-D

In [9]:
suppressPackageStartupMessages({
  if (!require(NbClust)) { install.packages("NbClust"); library(NbClust) }
  if (!require(mclust))  { install.packages("mclust");  library(mclust)  }
})

## ===== 1) CSV -> 厳密に数値行列へ =====
read_mds_numeric <- function(path) {
  df <- tryCatch({
    read.csv(path, check.names = FALSE, row.names = 1, stringsAsFactors = FALSE)
  }, error = function(e) stop(sprintf("read.csv failed: %s\n%s", path, e$message), call. = FALSE))

  # 列ごとに安全に numeric 化（"","NA"," " 等も NA 扱い）
  num <- as.data.frame(lapply(df, function(x) {
    if (is.numeric(x)) return(x)
    x[x %in% c("", " ", "NA", "NaN", "null")] <- NA
    as.numeric(x)
  }), check.names = FALSE)
  rownames(num) <- rownames(df)

  # 定数列は落とす
  if (ncol(num) > 0) {
    keep_col <- vapply(num, function(v) sd(v, na.rm = TRUE) > 0, logical(1))
    num <- num[, keep_col, drop = FALSE]
  }
  # 欠損行は落とす（行名は維持）
  if (nrow(num) > 0) {
    num <- num[stats::complete.cases(num), , drop = FALSE]
  }

  if (ncol(num) < 2) stop(sprintf("Numeric columns < 2 in %s", path), call. = FALSE)
  if (nrow(num) < 3) stop(sprintf("Numeric rows < 3 in %s", path), call. = FALSE)

  as.matrix(num)
}

## ===== 2) partition を X の行順に確実に整列 =====
align_partition <- function(cl, X) {
  rn <- rownames(X)
  out <- rep(NA_integer_, length(rn)); names(out) <- rn
  if (is.null(cl)) return(out)

  cl <- as.integer(cl)
  if (is.null(names(cl))) {
    # 名前が無い：長さ一致なら行順で付ける
    if (length(cl) == length(rn)) {
      names(cl) <- rn
    } else {
      return(out)
    }
  }
  # 足りない名前がある：長さ一致なら強制的に行順名
  if (!all(rn %in% names(cl))) {
    if (length(cl) == length(rn)) {
      names(cl) <- rn
    } else {
      return(out)
    }
  }
  cl[rn]
}

## ===== 3) NbClust の堅牢ラッパ =====
safe_nbclust <- function(X, index_name, max_k = 100) {
  # すべて数値・有限に
  stopifnot(is.matrix(X))
  if (!is.numeric(X)) storage.mode(X) <- "double"
  if (any(!is.finite(X))) {
    keep <- apply(X, 1, function(r) all(is.finite(r)))
    X <- X[keep, , drop = FALSE]
  }
  if (ncol(X) < 2L) stop("X has <2 numeric columns after cleaning", call. = FALSE)
  if (nrow(X) < 3L) stop("X has <3 rows after cleaning", call. = FALSE)

  max_nc <- min(max_k, nrow(X) - 1L)
  if (max_nc < 2L) stop("max_nc < 2 (too few rows)", call. = FALSE)

  suppressWarnings(tryCatch({
    NbClust(data = X, diss = NULL, distance = "euclidean",
            min.nc = 2, max.nc = max_nc,
            method = "ward.D2", index = index_name)
  }, error = function(e) {
    stop(sprintf("NbClust failed (index=%s): %s", index_name, e$message), call. = FALSE)
  }))
}

## ===== 4) 軸サブセットを作る（Top3/K80/D1-D2/D1-D3/D2-D3）=====
build_axis_subsets <- function(mtx_top3, mtx_k80) {
  # 列名が "Dim1","Dim2",... である前提。違う場合はここで正規化。
  cn_top <- colnames(mtx_top3); cn_k80 <- colnames(mtx_k80)
  # 念のため Dim1..DimN に揃える
  colnames(mtx_top3) <- paste0("Dim", seq_len(ncol(mtx_top3)))
  colnames(mtx_k80)  <- paste0("Dim", seq_len(ncol(mtx_k80)))

  common_ids <- intersect(rownames(mtx_top3), rownames(mtx_k80))
  mtx_top3 <- mtx_top3[common_ids, , drop = FALSE]
  mtx_k80  <- mtx_k80 [common_ids, , drop = FALSE]

  list(
    "Top3"  = mtx_top3[, 1:min(3, ncol(mtx_top3)), drop = FALSE],
    "K80"   = mtx_k80,
    "D1-D2" = mtx_k80[, 1:2, drop = FALSE],
    "D1-D3" = if (ncol(mtx_k80) >= 3) mtx_k80[, c(1,3), drop = FALSE] else NULL,
    "D2-D3" = if (ncol(mtx_k80) >= 3) mtx_k80[, 2:3, drop = FALSE] else NULL
  )
}

## ===== 5) 単発デバッグ用（まずエラー内容を出す）=====
debug_one <- function(top3_csv, k80_csv, index_name = "silhouette",
                      from_axis = "K80", to_axis = "Top3", max_k = 100) {
  cat("[MDS file]", top3_csv, "\n")
  cat("[MDS file]", k80_csv,  "\n")

  top3 <- read_mds_numeric(top3_csv)
  k80  <- read_mds_numeric(k80_csv)

  axes <- build_axis_subsets(top3, k80)
  if (is.null(axes[[from_axis]]) || is.null(axes[[to_axis]])) {
    stop(sprintf("axis not available: from=%s to=%s", from_axis, to_axis), call. = FALSE)
  }

  X1 <- axes[[from_axis]]
  X2 <- axes[[to_axis]]

  res1 <- safe_nbclust(X1, index_name, max_k = max_k)
  res2 <- safe_nbclust(X2, index_name, max_k = max_k)

  cl1  <- align_partition(res1$Best.partition, X1)
  cl2  <- align_partition(res2$Best.partition, X2)

  nn <- sum(!is.na(cl1) & !is.na(cl2))
  ari <- if (nn > 0) mclust::adjustedRandIndex(cl1[!is.na(cl1) & !is.na(cl2)],
                                               cl2[!is.na(cl2) & !is.na(cl1)]) else NA_real_
  cat(sprintf("n=%d, ARI=%.6f\n", nn, ari))
  invisible(list(n = nn, ARI = ari, cl1 = cl1, cl2 = cl2))
}

## ===== 6) 並列本番を回す前に、まず1ケースだけ通す =====
# 例：
# debug_one(
#   top3_csv = "/.../STEP3_new_MDS_20250904/OH/MDScoords_top3_OH_20250904_153003.csv",
#   k80_csv  = "/.../STEP3_new_MDS_20250904/OH/MDScoords_cumeig_OH_20250904_153003.csv",
#   index_name = "silhouette", from_axis = "K80", to_axis = "Top3", max_k = 100
# )


In [11]:
# 先にパッチで定義した関数（read_mds_numeric, debug_one など）がロード済みであること
if (!exists("debug_one")) stop("先にパッチ（関数定義）を実行してください。")

top3_csv <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/OH/MDScoords_top3_OH_20250904_153003.csv"
k80_csv  <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/OH/MDScoords_cumeig_OH_20250904_153003.csv"

cat("[EXISTS] top3:", file.exists(top3_csv), " / k80:", file.exists(k80_csv), "\n")

indices <- c("silhouette","dunn","gap","ch","db","ptbiserial")
for (ix in indices) {
  cat("\n==== NEW / OH :", ix, "(K80 vs Top3) ====\n")
  invisible(debug_one(top3_csv = top3_csv,
                      k80_csv  = k80_csv,
                      index_name = ix,
                      from_axis = "K80", to_axis = "Top3",
                      max_k = 100))
}


[EXISTS] top3: TRUE  / k80: TRUE 

==== NEW / OH : silhouette (K80 vs Top3) ====
[MDS file] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/OH/MDScoords_top3_OH_20250904_153003.csv 
[MDS file] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/OH/MDScoords_cumeig_OH_20250904_153003.csv 
n=394, ARI=0.971521

==== NEW / OH : dunn (K80 vs Top3) ====
[MDS file] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/OH/MDScoords_top3_OH_20250904_153003.csv 
[MDS file] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/OH/MDScoords_cumeig_OH_20250904_153003.csv 
n=394, ARI=0.651992

==== NEW / OH : gap (K80 vs Top3) ====
[MDS file] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/OH/MDScoords_top3_OH_20250904_153003.csv 
[MDS file] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/OH/MDScoords_cumeig_

In [12]:
top3_csv <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/FP/MDScoords_top3_FP_20250904_154030.csv"
k80_csv  <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/FP/MDScoords_cumeig_FP_20250904_154030.csv"

cat("[EXISTS] top3:", file.exists(top3_csv), " / k80:", file.exists(k80_csv), "\n")

indices <- c("silhouette","dunn","gap","ch","db","ptbiserial")
for (ix in indices) {
  cat("\n==== NEW / FP :", ix, "(K80 vs Top3) ====\n")
  invisible(debug_one(top3_csv = top3_csv,
                      k80_csv  = k80_csv,
                      index_name = ix,
                      from_axis = "K80", to_axis = "Top3",
                      max_k = 100))
}


[EXISTS] top3: TRUE  / k80: TRUE 

==== NEW / FP : silhouette (K80 vs Top3) ====
[MDS file] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/FP/MDScoords_top3_FP_20250904_154030.csv 
[MDS file] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/FP/MDScoords_cumeig_FP_20250904_154030.csv 
n=251, ARI=0.726981

==== NEW / FP : dunn (K80 vs Top3) ====
[MDS file] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/FP/MDScoords_top3_FP_20250904_154030.csv 
[MDS file] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/FP/MDScoords_cumeig_FP_20250904_154030.csv 
n=251, ARI=0.665931

==== NEW / FP : gap (K80 vs Top3) ====
[MDS file] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/FP/MDScoords_top3_FP_20250904_154030.csv 
[MDS file] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/FP/MDScoords_cumeig_

In [13]:
# ============== セットアップ ==============
suppressPackageStartupMessages({
  if (!requireNamespace("data.table", quietly=TRUE)) install.packages("data.table")
})
library(data.table)

# mclust があれば使う（なければ内蔵ARIを使用）
has_mclust <- requireNamespace("mclust", quietly = TRUE)

calc_ari <- function(x, y){
  # x, y はクラスタラベル（整数 or 文字）
  ok <- !(is.na(x) | is.na(y))
  x <- x[ok]; y <- y[ok]
  n <- length(x)
  if (n <= 1) return(list(n = n, ARI = NA_real_))
  if (has_mclust) {
    return(list(n = n, ARI = as.numeric(mclust::adjustedRandIndex(x, y))))
  }
  # --- フォールバック（自前 ARI） ---
  tab <- table(x, y)
  sum_ij <- sum(choose(tab, 2))
  a <- rowSums(tab); b <- colSums(tab)
  sum_a <- sum(choose(a, 2))
  sum_b <- sum(choose(b, 2))
  comb_n <- choose(n, 2)
  exp_idx <- if (comb_n == 0) 0 else (sum_a * sum_b / comb_n)
  max_idx <- 0.5 * (sum_a + sum_b)
  denom <- (max_idx - exp_idx)
  ARI <- if (denom == 0) NA_real_ else (sum_ij - exp_idx) / denom
  list(n = n, ARI = as.numeric(ARI))
}

# ============== パス設定（あなたのログに合わせて固定） ==============
root      <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
step2_dir <- file.path(root, "STEP2_old_MDS_20250904")                 # 使わない（参考）
new_dir   <- file.path(root, "STEP3_new_MDS_20250904")
sgn_dir   <- file.path(root, "STEP3.2_signlessCorr_MDS_20250904")
old_dir   <- file.path(root, "STEP3.3_olddata_MDS_20250904")

datasets  <- c("OH", "FP")

# ============== ユーティリティ ==============
latest_by_pattern <- function(dir, pattern){
  fs <- list.files(dir, pattern = pattern, full.names = TRUE)
  if (length(fs) == 0) return(NA_character_)
  info <- file.info(fs)
  fs[order(info$mtime, decreasing = TRUE)][1]
}

read_labels <- function(csv_path){
  if (is.na(csv_path) || !file.exists(csv_path)) {
    stop(sprintf("File not found: %s", csv_path), call. = FALSE)
  }
  dt <- fread(csv_path, header = TRUE)
  # id 列の推定：'id' があればそれ、なければ最左の非数値カラム
  id_col <- if ("id" %in% names(dt)) "id" else {
    nn <- names(dt)
    cand <- nn[!vapply(dt, is.numeric, logical(1))]
    if (length(cand) == 0) nn[1] else cand[1]
  }
  setnames(dt, id_col, "id")
  # ラベル列（整数/数値）だけを残す（id は別）
  lab_cols <- setdiff(names(dt), "id")
  # keep 数値/整数（NA 可）
  keep <- vapply(dt[, ..lab_cols], function(v) is.numeric(v) || is.integer(v), logical(1))
  dt <- cbind(dt[, .(id)], dt[, ..lab_cols][, ..lab_cols[keep]])
  # すべて整数に寄せる（クラスタラベル用途）
  for (cn in setdiff(names(dt), "id")){
    suppressWarnings(dt[[cn]] <- as.integer(round(dt[[cn]])))
  }
  dt[]
}

# signless の線形/非線形を列名から振り分け & 規格化
split_signless_cols <- function(colnames_vec){
  cn <- tolower(colnames_vec)
  # 非線形の判定（nl / nonlin / nonlinear を含む列）
  is_nl <- grepl("(^|[._-])(nl|nonlin|nonlinear)($|[._-])", cn, perl=TRUE)
  # ベース名（nl 等を取り除く）
  base <- gsub("(^|[._-])(nl|nonlin|nonlinear)($|[._-])", "_", cn, perl=TRUE)
  base <- gsub("^linear[._-]?", "", base, perl=TRUE)
  base <- gsub("[^a-z0-9_]+", "_", base, perl=TRUE)
  # 使う候補（top3/cumeig × 6指標）
  allowed <- c(outer(c("top3","cumeig"),
                     c("ch","db","dunn","gap","ptbiserial","silhouette"),
                     paste, sep = "_"))
  # mapping を返す（base名 -> 元の列名）
  map_lin <- setNames(colnames_vec[!is_nl], base[!is_nl])
  map_lin <- map_lin[names(map_lin) %in% allowed]
  map_nln <- setNames(colnames_vec[ is_nl], base[ is_nl])
  map_nln <- map_nln[names(map_nln) %in% allowed]
  list(linear = map_lin, nonlinear = map_nln)
}

# 共通 id でそろえつつ、共通の条件名だけ ARI を出す
compare_label_tables <- function(dtA, dtB, mapA, mapB, title){
  common_ids <- intersect(dtA$id, dtB$id)
  if (length(common_ids) == 0) {
    cat(title, " -> [no common ids]\n")
    return(invisible(NULL))
  }
  setkeyv(dtA, "id"); setkeyv(dtB, "id")
  A <- dtA[J(common_ids)]
  B <- dtB[J(common_ids)]

  conds <- intersect(names(mapA), names(mapB))
  if (length(conds) == 0) {
    cat(title, " -> [no common conditions]\n")
    return(invisible(NULL))
  }

  res <- lapply(conds, function(cd){
    a_col <- mapA[[cd]]
    b_col <- mapB[[cd]]
    xy <- calc_ari(A[[a_col]], B[[b_col]])
    data.frame(condition = cd, n = xy$n, ARI = xy$ARI, stringsAsFactors = FALSE)
  })
  out <- do.call(rbind, res)
  rownames(out) <- NULL
  # 並べ替え：cumeig_* → top3_*
  ord <- c(grep("^cumeig_", out$condition), grep("^top3_", out$condition))
  out <- out[ord, ]
  cat("\n", title, "\n", sep = "")
  print(out, row.names = FALSE)
  invisible(out)
}

# ============== メイン ==============
cat("Resolved root:", root, "\n")

for (ds in datasets){
  cat("\n================ DATASET:", ds, "================\n")

  # 各ステップの cluster_labels_* を最新1ファイルずつ取得
  new_csv <- latest_by_pattern(file.path(new_dir, ds), "^cluster_labels.*\\.csv$")
  sgn_csv <- latest_by_pattern(file.path(sgn_dir, ds), "^cluster_labels.*\\.csv$")
  old_csv <- latest_by_pattern(file.path(old_dir, ds), "^cluster_labels.*\\.csv$")

  cat("[FILES]\n",
      " new:      ", new_csv, "\n",
      " signless: ", sgn_csv, "\n",
      " olddata:  ", old_csv, "\n", sep = "")

  # 読み込み
  new_dt <- read_labels(new_csv)
  sgn_dt <- read_labels(sgn_csv)
  old_dt <- read_labels(old_csv)

  # signless の列振り分け
  spl <- split_signless_cols(setdiff(names(sgn_dt), "id"))
  sgn_map_lin  <- spl$linear
  sgn_map_nln  <- spl$nonlinear

  # new/old は列名をそのままベース名とみなす（lower/正規化）
  norm_base <- function(cols){
    b <- tolower(cols)
    b <- gsub("[^a-z0-9_]+", "_", b)
    names(cols) <- b
    # 使う候補に限定
    allowed <- c(outer(c("top3","cumeig"),
                       c("ch","db","dunn","gap","ptbiserial","silhouette"),
                       paste, sep = "_"))
    cols[names(cols) %in% allowed]
  }
  new_map <- norm_base(setdiff(names(new_dt), "id"))
  old_map <- norm_base(setdiff(names(old_dt), "id"))

  # ========== 比較（コンソール出力） ==========
  compare_label_tables(new_dt, sgn_dt, new_map, sgn_map_lin,  sprintf("-- NEW vs SIGNLESS (linear)  [%s] --", ds))
  compare_label_tables(new_dt, sgn_dt, new_map, sgn_map_nln,  sprintf("-- NEW vs SIGNLESS (nonlinear) [%s] --", ds))
  compare_label_tables(new_dt, old_dt, new_map, old_map,      sprintf("-- NEW vs OLDDATA            [%s] --", ds))
  compare_label_tables(sgn_dt, sgn_dt, sgn_map_lin, sgn_map_nln, sprintf("-- SIGNLESS (linear) vs (nonlinear) [%s] --", ds))
  compare_label_tables(old_dt, sgn_dt, old_map, sgn_map_lin,  sprintf("-- OLDDATA vs SIGNLESS (linear)   [%s] --", ds))
  compare_label_tables(old_dt, sgn_dt, old_map, sgn_map_nln,  sprintf("-- OLDDATA vs SIGNLESS (nonlinear)[%s] --", ds))
}

cat("\nDONE: All cross-step ARI tables printed to console.\n")



Attaching package: 'data.table'


The following objects are masked from 'package:dplyr':

    between, first, last




Resolved root: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No 

================ DATASET: OH ================
[FILES]
 new:      NA
 signless: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3.2_signlessCorr_MDS_20250904/OH/cluster_labels_OH_20250904_160436.csv
 olddata:  NA


ERROR: Error: File not found: NA


In [14]:
# =========================
#  cross-step ARI console (robust)
#    - cluster_labels_*.csv が無い場合は ClusterAssign_*.csv 群から自動統合
#    - 見つからない組はスキップしつつ残りを出力
# =========================
suppressPackageStartupMessages({
  if (!requireNamespace("data.table", quietly=TRUE)) install.packages("data.table")
})
library(data.table)

# mclust があれば ARI に使用、無ければフォールバック実装
has_mclust <- requireNamespace("mclust", quietly = TRUE)
calc_ari <- function(x, y){
  ok <- !(is.na(x) | is.na(y))
  x <- x[ok]; y <- y[ok]
  n <- length(x)
  if (n <= 1) return(list(n=n, ARI=NA_real_))
  if (has_mclust) return(list(n=n, ARI=as.numeric(mclust::adjustedRandIndex(x,y))))
  # --- fallback ARI ---
  tab <- table(x, y)
  sum_ij <- sum(choose(tab, 2))
  a <- rowSums(tab); b <- colSums(tab)
  sum_a <- sum(choose(a, 2)); sum_b <- sum(choose(b, 2))
  comb_n <- choose(n, 2)
  exp_idx <- if (comb_n == 0) 0 else (sum_a * sum_b / comb_n)
  max_idx <- 0.5 * (sum_a + sum_b)
  denom <- (max_idx - exp_idx)
  ARI <- if (denom == 0) NA_real_ else (sum_ij - exp_idx) / denom
  list(n=n, ARI=as.numeric(ARI))
}

# ===== パス =====
root      <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
new_dir   <- file.path(root, "STEP3_new_MDS_20250904")
sgn_dir   <- file.path(root, "STEP3.2_signlessCorr_MDS_20250904")
old_dir   <- file.path(root, "STEP3.3_olddata_MDS_20250904")
datasets  <- c("OH","FP")

# ===== ユーティリティ =====
latest_by_patterns <- function(dir, patterns){
  fs <- unlist(lapply(patterns, function(p)
    list.files(dir, pattern=p, full.names=TRUE, ignore.case=TRUE)))
  fs <- unique(fs)
  if (length(fs) == 0) return(NA_character_)
  info <- file.info(fs)
  fs[order(info$mtime, decreasing=TRUE)][1]
}

# id 推定
guess_id_col <- function(dt){
  if ("id" %in% names(dt)) return("id")
  # 最左の非数値カラム
  nn <- names(dt)
  cand <- nn[!vapply(dt, is.numeric, logical(1))]
  if (length(cand) == 0) nn[1] else cand[1]
}

# ラベル列推定（ClusterAssign_* の中身が色々でも耐える）
pick_label_col <- function(dt){
  # 名前優先
  for (nm in c("cluster","label","labels","cl","partition")) {
    if (nm %in% names(dt) && (is.numeric(dt[[nm]]) || is.integer(dt[[nm]]))) return(nm)
  }
  # 数値カラムから最もユニーク値が多い列を採用
  num_cols <- names(dt)[vapply(dt, function(v) is.numeric(v) || is.integer(v), logical(1))]
  if (length(num_cols) == 0) return(NA_character_)
  uniq_counts <- sapply(num_cols, function(cn) length(unique(dt[[cn]])))
  num_cols[which.max(uniq_counts)]
}

# ClusterAssign_*.csv 群 → 1つの label テーブルへ統合
assemble_labels_from_assign <- function(dir_ds){
  assigns <- list.files(dir_ds, pattern="^ClusterAssign_.*\\.csv$", full.names=TRUE, ignore.case=TRUE)
  if (length(assigns) == 0) {
    message(sprintf("[assemble] no ClusterAssign_*.csv in %s", dir_ds))
    return(NULL)
  }
  message(sprintf("[assemble] found %d files under %s", length(assigns), dir_ds))
  allowed_idx <- c("ch","db","dunn","gap","ptbiserial","silhouette")
  build_one <- function(f){
    base <- basename(f); low <- tolower(base)
    mode <- if (grepl("top3", low)) "top3"
      else if (grepl("cumeig|k80|80", low)) "cumeig"
      else NA_character_
    idx <- NA_character_
    for (cand in allowed_idx) if (grepl(paste0("(^|[_-])",cand,"([_.-]|$)"), low)) { idx <- cand; break }
    if (is.na(mode) || is.na(idx)) {
      message(sprintf("  [skip] cannot parse mode/index from: %s", base))
      return(NULL)
    }
    cond <- paste(mode, idx, sep="_")
    dt <- tryCatch(data.table::fread(f, header=TRUE), error=function(e) NULL)
    if (is.null(dt) || nrow(dt)==0) {
      message(sprintf("  [skip] empty/unreadable: %s", base)); return(NULL)
    }
    idc <- guess_id_col(dt)
    setnames(dt, idc, "id")
    labc <- pick_label_col(dt)
    if (is.na(labc)) {
      message(sprintf("  [skip] no numeric label column in: %s", base)); return(NULL)
    }
    out <- unique(dt[, .(id, lab = as.integer(round(get(labc))))])
    setnames(out, "lab", cond)
    out
  }
  parts <- Filter(Negate(is.null), lapply(assigns, build_one))
  if (length(parts) == 0) return(NULL)
  # 逐次マージ
  out <- Reduce(function(a,b) merge(a,b,by="id",all=TRUE), parts)
  # id に NA が無い行のみ
  out <- out[!is.na(id)]
  out[]
}

# cluster_labels_* を読む（無ければ assemble にフォールバック）
read_labels <- function(csv_path, dir_ds){
  if (!is.na(csv_path) && file.exists(csv_path)) {
    dt <- fread(csv_path, header=TRUE)
    idc <- guess_id_col(dt)
    setnames(dt, idc, "id")
    # 数値ラベルだけ残す
    lab_cols <- setdiff(names(dt), "id")
    keep <- vapply(dt[, ..lab_cols], function(v) is.numeric(v) || is.integer(v), logical(1))
    dt <- cbind(dt[, .(id)], dt[, ..lab_cols][, ..lab_cols[keep]])
    for (cn in setdiff(names(dt), "id")) dt[[cn]] <- as.integer(round(dt[[cn]]))
    return(dt[])
  }
  # フォールバック: ClusterAssign_* から組み立て
  message(sprintf("[fallback] cluster_labels_*.csv not found. Try assembling from ClusterAssign_* in %s", dir_ds))
  assembled <- assemble_labels_from_assign(dir_ds)
  if (!is.null(assembled)) return(assembled[])
  message(sprintf("[fallback] failed to assemble labels in %s", dir_ds))
  NULL
}

# signless 列を linear / nonlinear に振り分け
split_signless_cols <- function(colnames_vec){
  cn <- tolower(colnames_vec)
  is_nl <- grepl("(^|[._-])(nl|nonlin|nonlinear)($|[._-])", cn, perl=TRUE)
  base <- gsub("(^|[._-])(nl|nonlin|nonlinear)($|[._-])", "_", cn, perl=TRUE)
  base <- gsub("^linear[._-]?", "", base, perl=TRUE)
  base <- gsub("[^a-z0-9_]+", "_", base, perl=TRUE)
  allowed <- c(outer(c("top3","cumeig"),
                     c("ch","db","dunn","gap","ptbiserial","silhouette"),
                     paste, sep="_"))
  map_lin <- setNames(colnames_vec[!is_nl], base[!is_nl]); map_lin <- map_lin[names(map_lin) %in% allowed]
  map_nln <- setNames(colnames_vec[ is_nl], base[ is_nl]); map_nln <- map_nln[names(map_nln) %in% allowed]
  list(linear=map_lin, nonlinear=map_nln)
}

norm_base <- function(cols){
  b <- tolower(cols); b <- gsub("[^a-z0-9_]+", "_", b); names(cols) <- b
  allowed <- c(outer(c("top3","cumeig"),
                     c("ch","db","dunn","gap","ptbiserial","silhouette"),
                     paste, sep="_"))
  cols[names(cols) %in% allowed]
}

compare_label_tables <- function(dtA, dtB, mapA, mapB, title){
  if (is.null(dtA) || is.null(dtB)) {
    cat(title, " -> [skip: missing table]\n"); return(invisible(NULL))
  }
  common_ids <- intersect(dtA$id, dtB$id)
  if (length(common_ids) == 0) { cat(title, " -> [no common ids]\n"); return(invisible(NULL)) }
  setkeyv(dtA, "id"); setkeyv(dtB, "id")
  A <- dtA[J(common_ids)]; B <- dtB[J(common_ids)]
  conds <- intersect(names(mapA), names(mapB))
  if (length(conds) == 0) { cat(title, " -> [no common conditions]\n"); return(invisible(NULL)) }
  res <- lapply(conds, function(cd){
    a_col <- mapA[[cd]]; b_col <- mapB[[cd]]
    xy <- calc_ari(A[[a_col]], B[[b_col]])
    data.frame(condition=cd, n=xy$n, ARI=xy$ARI, stringsAsFactors=FALSE)
  })
  out <- do.call(rbind, res); rownames(out) <- NULL
  ord <- c(grep("^cumeig_", out$condition), grep("^top3_", out$condition))
  out <- out[ord, ]
  cat("\n", title, "\n", sep="")
  print(out, row.names=FALSE)
  invisible(out)
}

# ===== メイン =====
cat("Resolved root:", root, "\n")

for (ds in datasets){
  cat("\n================ DATASET:", ds, "================\n")
  dir_new <- file.path(new_dir, ds)
  dir_sgn <- file.path(sgn_dir, ds)
  dir_old <- file.path(old_dir, ds)

  # 候補一覧（見つからない時のトラブルシュート用のログ出し）
  cand_list <- function(d){
    if (!dir.exists(d)) return(character(0))
    list.files(d, full.names=FALSE)
  }

  new_csv <- latest_by_patterns(dir_new, c("^cluster_labels.*\\.csv$", paste0("^.*", ds, ".*cluster.*labels.*\\.csv$")))
  sgn_csv <- latest_by_patterns(dir_sgn, c("^cluster_labels.*\\.csv$", paste0("^.*", ds, ".*cluster.*labels.*\\.csv$")))
  old_csv <- latest_by_patterns(dir_old, c("^cluster_labels.*\\.csv$", paste0("^.*", ds, ".*cluster.*labels.*\\.csv$")))

  cat("[FILES]\n",
      " new:      ", ifelse(is.na(new_csv),"NA",new_csv), "\n",
      " signless: ", ifelse(is.na(sgn_csv),"NA",sgn_csv), "\n",
      " olddata:  ", ifelse(is.na(old_csv),"NA",old_csv), "\n", sep="")
  if (is.na(new_csv))  cat("[hint] files in NEW     :", paste(cand_list(dir_new), collapse=", "), "\n")
  if (is.na(old_csv))  cat("[hint] files in OLDDATA :", paste(cand_list(dir_old), collapse=", "), "\n")
  if (is.na(sgn_csv))  cat("[hint] files in SIGNLESS:", paste(cand_list(dir_sgn), collapse=", "), "\n")

  new_dt <- read_labels(new_csv, dir_new)
  sgn_dt <- read_labels(sgn_csv, dir_sgn)
  old_dt <- read_labels(old_csv, dir_old)

  # マッピング
  spl <- if (!is.null(sgn_dt)) split_signless_cols(setdiff(names(sgn_dt), "id")) else list(linear=character(0), nonlinear=character(0))
  sgn_map_lin  <- spl$linear
  sgn_map_nln  <- spl$nonlinear
  new_map <- if (!is.null(new_dt)) norm_base(setdiff(names(new_dt), "id")) else character(0)
  old_map <- if (!is.null(old_dt)) norm_base(setdiff(names(old_dt), "id")) else character(0)

  # 比較
  compare_label_tables(new_dt, sgn_dt, new_map, sgn_map_lin,  sprintf("-- NEW vs SIGNLESS (linear)    [%s] --", ds))
  compare_label_tables(new_dt, sgn_dt, new_map, sgn_map_nln,  sprintf("-- NEW vs SIGNLESS (nonlinear) [%s] --", ds))
  compare_label_tables(new_dt, old_dt, new_map, old_map,      sprintf("-- NEW vs OLDDATA              [%s] --", ds))
  compare_label_tables(sgn_dt, sgn_dt, sgn_map_lin, sgn_map_nln, sprintf("-- SIGNLESS (linear) vs (nonlinear) [%s] --", ds))
  compare_label_tables(old_dt, sgn_dt, old_map, sgn_map_lin,  sprintf("-- OLDDATA vs SIGNLESS (linear)   [%s] --", ds))
  compare_label_tables(old_dt, sgn_dt, old_map, sgn_map_nln,  sprintf("-- OLDDATA vs SIGNLESS (nonlinear)[%s] --", ds))
}

cat("\nDONE: All cross-step ARI tables printed to console.\n")


Resolved root: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No 

================ DATASET: OH ================
[FILES]
 new:      NA
 signless: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3.2_signlessCorr_MDS_20250904/OH/cluster_labels_OH_20250904_160436.csv
 olddata:  NA
[hint] files in NEW     : ARI_top3_vs_cumeig_OH_20250904_153003.csv, ClusterAssign_cumeig_ch_OH_20250904_153003.csv, ClusterAssign_cumeig_db_OH_20250904_153003.csv, ClusterAssign_cumeig_dunn_OH_20250904_153003.csv, ClusterAssign_cumeig_gap_OH_20250904_153003.csv, ClusterAssign_cumeig_ptbiserial_OH_20250904_153003.csv, ClusterAssign_cumeig_silhouette_OH_20250904_153003.csv, ClusterAssign_top3_ch_OH_20250904_153003.csv, ClusterAssign_top3_db_OH_20250904_153003.csv, ClusterAssign_top3_dunn_OH_20250904_153003.csv, ClusterAssign_top3_gap_OH_20250904_153003.csv, ClusterAssign_top3_ptbiserial_OH_20250904_153003.csv, ClusterAssign_top3_silhouette_OH_20250904_153003.csv, Cluster_sizes_ch_OH_2025

[fallback] cluster_labels_*.csv not found. Try assembling from ClusterAssign_* in /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/OH

[assemble] found 12 files under /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/OH

Warning message in as.data.table.list(x, keep.rownames = keep.rownames, check.names = check.names, :
"Item 2 has 24 rows but longest item has 394; recycled with remainder."


ERROR: Error in round(dt[[cn]]): non-numeric argument to mathematical function


In [16]:
# ===== 追加: cluster_labels の欠落を報告する関数 =====
report_missing <- function(stage, ds, dir_stage, matched_path) {
  pat_labels <- "^cluster_labels.*\\.csv$"
  if (!dir.exists(dir_stage)) {
    cat(sprintf("[MISSING] %s/%s: directory not found: %s\n",
                stage, ds, dir_stage))
    return(invisible(NULL))
  }
  has_labels <- length(list.files(dir_stage, pattern=pat_labels,
                                  ignore.case=TRUE, full.names=TRUE)) > 0
  # matched_path が NA だったり、ディレクトリ内に1つも見つからない場合に出力
  if (is.na(matched_path) || !has_labels) {
    assigns <- list.files(dir_stage, pattern="^ClusterAssign_.*\\.csv$",
                          ignore.case=TRUE, full.names=TRUE)
    cat(sprintf("[MISSING] %s/%s: cluster_labels_*.csv NOT found under: %s\n",
                stage, ds, dir_stage))
    cat(sprintf("          fallback: %d × ClusterAssign_*.csv %s\n",
                length(assigns),
                ifelse(length(assigns) > 0, "available", "NOT available")))
  }
  invisible(NULL)
}

# 既存の [FILES] 出力の直後に追加
report_missing("NEW",      ds, dir_new, new_csv)
report_missing("SIGNLESS", ds, dir_sgn, sgn_csv)
report_missing("OLDDATA",  ds, dir_old, old_csv)


[MISSING] NEW/OH: cluster_labels_*.csv NOT found under: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/OH
          fallback: 12 <U+00D7> ClusterAssign_*.csv available
[MISSING] OLDDATA/OH: cluster_labels_*.csv NOT found under: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3.3_olddata_MDS_20250904/OH
          fallback: 12 <U+00D7> ClusterAssign_*.csv available


In [17]:
# =========================
# ClusterAssign_* → cluster_labels_*.csv 合成スクリプト
# 対象: STEP3_new_MDS_YYYYMMDD, STEP3.3_olddata_MDS_YYYYMMDD（OH/FP）
# 出力: フォルダ直下に cluster_labels_<DATASET>_<YYYYMMDD_HHMMSS>.csv
# =========================

suppressPackageStartupMessages({
  if (!requireNamespace("data.table", quietly = TRUE)) install.packages("data.table")
  library(data.table)
})

# ---- ルートと対象フォルダ（必要に応じて変更） ----
root <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
steps <- c(
  NEW     = file.path(root, "STEP3_new_MDS_20250904"),
  OLDDATA = file.path(root, "STEP3.3_olddata_MDS_20250904")
)
datasets <- c("OH","FP")

# 列名推定に使う既知のインデックス語彙
known_idx <- c("silhouette","dunn","gap","ch","db","ptbiserial","cindex","mcclain","frey")
known_mode <- c("top3","cumeig")

# 列の表示順（お好みで並び替え）
order_index <- c("silhouette","dunn","gap","ch","db","ptbiserial","cindex","mcclain","frey")

# ---------------- ユーティリティ ----------------
norm_colname <- function(fname) {
  # ファイル名から mode と index を推定して "mode_index" を返す
  f <- tolower(basename(fname))
  mode <- NA_character_
  for (m in known_mode) if (grepl(paste0("(^|_|-)", m, "($|_|\\.)"), f)) { mode <- m; break }
  idx  <- NA_character_
  for (k in known_idx) if (grepl(paste0("(^|_|-)", k, "($|_|\\.)"), f)) { idx <- k; break }
  list(mode = mode, idx = idx,
       col  = ifelse(!is.na(mode) && !is.na(idx), paste0(mode, "_", idx), NA_character_))
}

read_assign <- function(path) {
  # ClusterAssign_* を読み、id列・クラスタ列を標準化
  dt <- suppressWarnings(fread(path))
  if (ncol(dt) == 0) stop(sprintf("Empty file: %s", path), call. = FALSE)

  # id列の推定
  id_candidates <- c("id","ID","feature","name","var","variable")
  id_col <- id_candidates[id_candidates %in% names(dt)]
  if (length(id_col) == 0) id_col <- names(dt)[1] else id_col <- id_col[1]

  # ラベル列の推定（数値or整数っぽい列を優先。無ければ2列目）
  other_cols <- setdiff(names(dt), id_col)
  num_cols <- other_cols[vapply(dt[, ..other_cols], function(x) is.numeric(x) || is.integer(x), logical(1))]
  lab_col <- if (length(num_cols) >= 1) num_cols[1] else {
    if (length(other_cols) >= 1) other_cols[1] else stop(sprintf("No label column in %s", path), call. = FALSE)
  }

  out <- dt[, .(id = as.character(get(id_col)), value = as.integer(as.character(get(lab_col))))]
  # クラスタが因子/文字のケースに備えて数値化（文字→整数連番）
  if (any(is.na(out$value))) {
    # 文字 → factor → 整数
    out[, value := as.integer(factor(dt[[lab_col]]))]
  }
  out
}

merge_labels <- function(dts) {
  Reduce(function(x, y) merge(x, y, by = "id", all = TRUE), dts)
}

order_label_cols <- function(nms) {
  # mode（cumeig→top3 の順）かつ index の既定順で並べる
  spl <- tstrsplit(nms, "_", fixed = TRUE)
  md  <- factor(spl[[1]], levels = c("cumeig","top3"), ordered = TRUE)
  ix  <- factor(spl[[2]], levels = order_index, ordered = TRUE)
  nms[order(md, ix, nms)]
}

# ---------------- メイン処理 ----------------
build_one <- function(step_name, step_dir, dataset) {
  cat(sprintf("\n[%s / %s]\nDIR = %s/%s\n", step_name, dataset, step_dir, dataset))
  ddir <- file.path(step_dir, dataset)
  if (!dir.exists(ddir)) { cat("  ! dataset dir not found. skip.\n"); return(invisible(NULL)) }

  # 既に cluster_labels_* があるなら知らせてスキップ（上書きしたい場合は条件を変える）
  existing <- list.files(ddir, pattern = "^cluster_labels_.*\\.csv$", full.names = TRUE)
  if (length(existing) > 0) {
    cat(sprintf("  - cluster_labels_* already exists (%d file). skip making new.\n", length(existing)))
    return(invisible(existing))
  }

  files <- list.files(ddir, pattern = "^ClusterAssign_.*\\.csv$", full.names = TRUE)
  if (length(files) == 0) { cat("  ! ClusterAssign_*.csv not found. skip.\n"); return(invisible(NULL)) }
  cat(sprintf("  - found %d ClusterAssign_*.csv\n", length(files)))

  # 各ファイルの列名推定
  meta <- lapply(files, norm_colname)
  cols <- vapply(meta, `[[`, character(1), "col")
  modes<- vapply(meta, `[[`, character(1), "mode")
  idxs <- vapply(meta, `[[`, character(1), "idx")

  # 解析不能なファイルのレポート
  bad <- which(is.na(cols))
  if (length(bad) > 0) {
    cat("  ! WARNING: could not parse mode/index from file name:\n")
    for (i in bad) cat(sprintf("    - %s\n", basename(files[i])))
    cat("    → これらは除外されます（必要なら命名規則に top3/cumeig と指標名を含めてください）\n")
  }
  keep <- setdiff(seq_along(files), bad)
  files <- files[keep]; cols <- cols[keep]

  # 読み込み
  dts <- vector("list", length(files))
  for (i in seq_along(files)) {
    nm <- cols[i]
    dt <- read_assign(files[i])
    setnames(dt, "value", nm)
    dts[[i]] <- dt
    cat(sprintf("    + %-35s  →  col=%s (n=%d)\n", basename(files[i]), nm, nrow(dt)))
  }

  # マージ（idで外部結合）
  res <- merge_labels(dts)
  setcolorder(res, c("id", order_label_cols(setdiff(names(res), "id"))))
  cat(sprintf("  - merged: %d ids × %d label columns\n", nrow(res), ncol(res)-1))

  # 保存
  ts <- format(Sys.time(), "%Y%m%d_%H%M%S")
  out <- file.path(ddir, sprintf("cluster_labels_%s_%s.csv", dataset, ts))
  fwrite(res, out)
  cat(sprintf("  [SAVED] %s\n", out))
  invisible(out)
}

# 実行
for (step_name in names(steps)) {
  step_dir <- steps[[step_name]]
  if (!dir.exists(step_dir)) { cat(sprintf("\n[%s] not found: %s\n", step_name, step_dir)); next }
  for (ds in datasets) build_one(step_name, step_dir, ds)
}

cat("\nDONE: cluster_labels files generation completed (existing files were kept).\n")



[NEW / OH]
DIR = /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/OH
  - found 12 ClusterAssign_*.csv
    + ClusterAssign_cumeig_ch_OH_20250904_153003.csv  <U+2192>  col=cumeig_ch (n=394)
    + ClusterAssign_cumeig_db_OH_20250904_153003.csv  <U+2192>  col=cumeig_db (n=394)
    + ClusterAssign_cumeig_dunn_OH_20250904_153003.csv  <U+2192>  col=cumeig_dunn (n=394)
    + ClusterAssign_cumeig_gap_OH_20250904_153003.csv  <U+2192>  col=cumeig_gap (n=394)
    + ClusterAssign_cumeig_ptbiserial_OH_20250904_153003.csv  <U+2192>  col=cumeig_ptbiserial (n=394)
    + ClusterAssign_cumeig_silhouette_OH_20250904_153003.csv  <U+2192>  col=cumeig_silhouette (n=394)
    + ClusterAssign_top3_ch_OH_20250904_153003.csv  <U+2192>  col=top3_ch (n=394)
    + ClusterAssign_top3_db_OH_20250904_153003.csv  <U+2192>  col=top3_db (n=394)
    + ClusterAssign_top3_dunn_OH_20250904_153003.csv  <U+2192>  col=top3_dunn (n=394)
    + ClusterAssign_top3_gap_OH_20250904_153003.csv  <U+219

In [18]:
# =============================================
# クラスター比較（ARI）スクリプト
# 比較:
#   1) NEW  vs SIGNLESS(linear)
#   2) NEW  vs OLDDATA
#   3) SIGNLESS: linear vs nonlinear（同一指標）
#   4) 各ステップ内: cumeig vs top3（同一指標）
# 入力: 各 step 配下の cluster_labels_*.csv（id列 + 各ラベル列）
# 出力: コンソール表示（必要なら write.csv 部分を追加して保存可）
# =============================================

suppressPackageStartupMessages({
  if (!requireNamespace("data.table", quietly = TRUE)) install.packages("data.table")
  if (!requireNamespace("mclust", quietly = TRUE)) install.packages("mclust")
  library(data.table)
  library(mclust)  # adjustedRandIndex
})

# ルートと step パス（必要に応じて調整）
root <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
paths <- list(
  step2    = file.path(root, "STEP2_old_MDS_20250904"),            # 参照のみ（labels無ければスキップ）
  new      = file.path(root, "STEP3_new_MDS_20250904"),
  signless = file.path(root, "STEP3.2_signlessCorr_MDS_20250904"),
  olddata  = file.path(root, "STEP3.3_olddata_MDS_20250904")
)
datasets <- c("OH","FP")

# --------- ユーティリティ ---------
latest_labels <- function(dir_ds) {
  # dir_ds = <step_dir>/<dataset> のパス
  if (!dir.exists(dir_ds)) return(NULL)
  fs <- list.files(dir_ds, pattern = "^cluster_labels_.*\\.csv$", full.names = TRUE)
  if (length(fs) == 0) return(NULL)
  fs[which.max(file.mtime(fs))]
}

read_labels_plain <- function(f) {
  # id + ラベル列（new/olddata/step2 想定）
  dt <- fread(f)
  if (!"id" %in% names(dt)) setnames(dt, 1, "id")
  dt[, id := as.character(id)]
  # 数値化（文字ラベルは連番化）
  for (cn in setdiff(names(dt), "id")) {
    x <- dt[[cn]]
    if (!(is.integer(x) || is.numeric(x))) dt[[cn]] <- as.integer(factor(x))
  }
  dt
}

read_labels_signless <- function(f) {
  # signless 用: linear_*, nonlinear_* を分離して返す
  dt <- fread(f)
  if (!"id" %in% names(dt)) setnames(dt, 1, "id")
  dt[, id := as.character(id)]

  nms <- setdiff(names(dt), "id")
  lin <- nms[grepl("^linear_", nms)]
  non <- nms[grepl("^nonlinear_", nms)]
  oth <- setdiff(nms, c(lin, non))  # 接頭辞なし（稀）

  # 文字ラベルは連番化
  as_int <- function(v) {
    if (is.integer(v) || is.numeric(v)) return(as.integer(v))
    as.integer(factor(v))
  }

  # linear
  dt_lin <- if (length(lin) > 0) {
    tmp <- data.table(id = dt$id)
    for (cn in lin) tmp[[sub("^linear_", "", cn)]] <- as_int(dt[[cn]])
    tmp
  } else NULL

  # nonlinear
  dt_non <- if (length(non) > 0) {
    tmp <- data.table(id = dt$id)
    for (cn in non) tmp[[sub("^nonlinear_", "", cn)]] <- as_int(dt[[cn]])
    tmp
  } else NULL

  # prefix なし（あれば linear 扱いに寄せる）
  if (length(oth) > 0) {
    if (is.null(dt_lin)) dt_lin <- data.table(id = dt$id)
    for (cn in oth) {
      newcn <- cn
      # top3_linear のような命名があれば整える
      newcn <- sub("_linear$", "", newcn)
      dt_lin[[newcn]] <- as_int(dt[[cn]])
    }
  }

  list(linear = dt_lin, nonlinear = dt_non)
}

# 共通 id での ARI を指標ごとに計算
compute_ari_table <- function(A, B, title = NULL) {
  if (is.null(A) || is.null(B)) {
    cat(sprintf("%s (no data)\n", ifelse(is.null(title), "[no title]", title)))
    return(invisible(NULL))
  }
  colsA <- setdiff(names(A), "id")
  colsB <- setdiff(names(B), "id")
  common <- intersect(colsA, colsB)
  if (length(common) == 0) {
    cat(sprintf("%s (no common columns)\n", ifelse(is.null(title), "[no title]", title)))
    return(invisible(NULL))
  }
  # id 合流
  merged <- merge(A[, c("id", common), with = FALSE],
                  B[, c("id", common), with = FALSE],
                  by = "id", all = FALSE, suffixes = c(".A",".B"))
  if (nrow(merged) == 0) {
    cat(sprintf("%s (no overlapping ids)\n", ifelse(is.null(title), "[no title]", title)))
    return(invisible(NULL))
  }
  out <- data.table(condition = character(), n = integer(), ARI = numeric())
  for (cn in common) {
    x <- merged[[paste0(cn, ".A")]]
    y <- merged[[paste0(cn, ".B")]]
    n <- sum(!is.na(x) & !is.na(y))
    ari <- if (n >= 2 && data.table::uniqueN(x) > 1 && data.table::uniqueN(y) > 1) {
      adjustedRandIndex(x, y)
    } else NA_real_
    out <- rbind(out, data.table(condition = cn, n = n, ARI = ari))
  }
  # 並びを見やすく（cumeig_* → top3_* の順）
  md  <- ifelse(grepl("^cumeig_", out$condition), 1L,
                ifelse(grepl("^top3_", out$condition), 2L, 3L))
  out <- out[order(md, condition)]
  cat(sprintf("\n%s\n", ifelse(is.null(title), "[Comparison]", title)))
  print(out, row.names = FALSE)
  invisible(out)
}

# 同一ステップ内の cumeig vs top3（各指標）の比較
compare_cumeig_vs_top3 <- function(DT, title) {
  if (is.null(DT)) { cat(sprintf("\n%s (no data)\n", title)); return(invisible(NULL)) }
  cols <- setdiff(names(DT), "id")
  idxs <- unique(sub("^(cumeig|top3)_", "", cols))
  idxs <- idxs[idxs != ""]
  out <- data.table(index = character(), n = integer(), ARI = numeric())
  for (ix in idxs) {
    c1 <- paste0("cumeig_", ix)
    c2 <- paste0("top3_", ix)
    if (c1 %in% cols && c2 %in% cols) {
      merged <- DT[, .(id, x = get(c1), y = get(c2))]
      merged <- merged[!is.na(x) & !is.na(y)]
      n <- nrow(merged)
      ari <- if (n >= 2 && uniqueN(merged$x) > 1 && uniqueN(merged$y) > 1) adjustedRandIndex(merged$x, merged$y) else NA_real_
      out <- rbind(out, data.table(index = ix, n = n, ARI = ari))
    }
  }
  if (nrow(out) == 0) {
    cat(sprintf("\n%s (no cumeig/top3 pairs)\n", title))
  } else {
    cat(sprintf("\n%s\n", title))
    print(out[order(index)], row.names = FALSE)
  }
  invisible(out)
}

# --------- 実行 ---------
for (ds in datasets) {
  cat(sprintf("\n================ DATASET: %s ================\n", ds))

  # 各 step の cluster_labels を読む
  f_new      <- latest_labels(file.path(paths$new,      ds))
  f_old      <- latest_labels(file.path(paths$olddata,  ds))
  f_signless <- latest_labels(file.path(paths$signless, ds))
  f_step2    <- latest_labels(file.path(paths$step2,    ds))  # あれば読む（無ければ NULL）

  cat("[FILES]\n")
  cat(sprintf("  new:      %s\n", ifelse(is.null(f_new), "NA", f_new)))
  cat(sprintf("  signless: %s\n", ifelse(is.null(f_signless), "NA", f_signless)))
  cat(sprintf("  olddata:  %s\n", ifelse(is.null(f_old), "NA", f_old)))
  if (!is.null(f_step2)) cat(sprintf("  step2:    %s\n", f_step2))

  NEW  <- if (!is.null(f_new)) read_labels_plain(f_new) else NULL
  OLD  <- if (!is.null(f_old)) read_labels_plain(f_old) else NULL
  SL   <- if (!is.null(f_signless)) read_labels_signless(f_signless) else NULL
  SL_L <- if (!is.null(SL)) SL$linear else NULL
  SL_N <- if (!is.null(SL)) SL$nonlinear else NULL
  STEP2<- if (!is.null(f_step2)) read_labels_plain(f_step2) else NULL

  # 1) NEW vs SIGNLESS(linear)
  compute_ari_table(NEW, SL_L, sprintf("-- NEW vs SIGNLESS(linear) [%s] --", ds))

  # 2) NEW vs OLDDATA
  compute_ari_table(NEW, OLD, sprintf("-- NEW vs OLDDATA [%s] --", ds))

  # 3) SIGNLESS: linear vs nonlinear
  compute_ari_table(SL_L, SL_N, sprintf("-- SIGNLESS: linear vs nonlinear [%s] --", ds))

  # 4) 同一ステップ内: cumeig vs top3
  compare_cumeig_vs_top3(NEW,  sprintf("-- Within NEW: cumeig vs top3 [%s] --", ds))
  compare_cumeig_vs_top3(OLD,  sprintf("-- Within OLDDATA: cumeig vs top3 [%s] --", ds))
  compare_cumeig_vs_top3(SL_L, sprintf("-- Within SIGNLESS(linear): cumeig vs top3 [%s] --", ds))
  compare_cumeig_vs_top3(SL_N, sprintf("-- Within SIGNLESS(nonlinear): cumeig vs top3 [%s] --", ds))
}

cat("\nDONE: All ARI comparisons printed to console.\n")



================ DATASET: OH ================
[FILES]
  new:      /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/OH/cluster_labels_OH_20250910_093155.csv
  signless: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3.2_signlessCorr_MDS_20250904/OH/cluster_labels_OH_20250904_160436.csv
  olddata:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3.3_olddata_MDS_20250904/OH/cluster_labels_OH_20250910_093156.csv

-- NEW vs SIGNLESS(linear) [OH] --
         condition     n        ARI
            <char> <int>      <num>
         cumeig_ch   394 0.72624273
         cumeig_db   394 0.02327565
       cumeig_dunn   394 0.77094799
        cumeig_gap   394 0.72624273
 cumeig_ptbiserial   394 0.09572465
 cumeig_silhouette   394 0.01917567
           top3_ch   394 0.24315860
           top3_db   394 0.24510104
         top3_dunn   394 0.94432851
          top3_gap   394 0.94432851
   top3_ptbiserial   394 0.50706218
   top3_silhouet

In [19]:
# =============================================
# クラスター比較（ARI）スクリプト - k表示付き
# 比較:
#   1) NEW  vs SIGNLESS(linear)
#   2) NEW  vs OLDDATA
#   3) SIGNLESS: linear vs nonlinear（同一指標）
#   4) 各ステップ内: cumeig vs top3（同一指標）
# 入力: 各 step 配下の cluster_labels_*.csv（id列 + 各ラベル列）
# 出力: コンソール表示（kA/kB = 片側ごとのクラスタ数）
# =============================================

suppressPackageStartupMessages({
  if (!requireNamespace("data.table", quietly = TRUE)) install.packages("data.table")
  if (!requireNamespace("mclust", quietly = TRUE)) install.packages("mclust")
  library(data.table)
  library(mclust)  # adjustedRandIndex
})

# ルートと step パス（必要に応じて調整）
root <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
paths <- list(
  step2    = file.path(root, "STEP2_old_MDS_20250904"),
  new      = file.path(root, "STEP3_new_MDS_20250904"),
  signless = file.path(root, "STEP3.2_signlessCorr_MDS_20250904"),
  olddata  = file.path(root, "STEP3.3_olddata_MDS_20250904")
)
datasets <- c("OH","FP")

# --------- ユーティリティ ---------
latest_labels <- function(dir_ds) {
  if (!dir.exists(dir_ds)) return(NULL)
  fs <- list.files(dir_ds, pattern = "^cluster_labels_.*\\.csv$", full.names = TRUE)
  if (length(fs) == 0) return(NULL)
  fs[which.max(file.mtime(fs))]
}

read_labels_plain <- function(f) {
  dt <- fread(f)
  if (!"id" %in% names(dt)) setnames(dt, 1, "id")
  dt[, id := as.character(id)]
  for (cn in setdiff(names(dt), "id")) {
    x <- dt[[cn]]
    if (!(is.integer(x) || is.numeric(x))) dt[[cn]] <- as.integer(factor(x))
  }
  dt
}

read_labels_signless <- function(f) {
  dt <- fread(f)
  if (!"id" %in% names(dt)) setnames(dt, 1, "id")
  dt[, id := as.character(id)]

  nms <- setdiff(names(dt), "id")
  lin <- nms[grepl("^linear_", nms)]
  non <- nms[grepl("^nonlinear_", nms)]
  oth <- setdiff(nms, c(lin, non))

  as_int <- function(v) if (is.integer(v) || is.numeric(v)) as.integer(v) else as.integer(factor(v))

  dt_lin <- if (length(lin) > 0) {
    tmp <- data.table(id = dt$id)
    for (cn in lin) tmp[[sub("^linear_", "", cn)]] <- as_int(dt[[cn]])
    tmp
  } else NULL

  dt_non <- if (length(non) > 0) {
    tmp <- data.table(id = dt$id)
    for (cn in non) tmp[[sub("^nonlinear_", "", cn)]] <- as_int(dt[[cn]])
    tmp
  } else NULL

  if (length(oth) > 0) {
    if (is.null(dt_lin)) dt_lin <- data.table(id = dt$id)
    for (cn in oth) {
      newcn <- sub("_linear$", "", cn)
      dt_lin[[newcn]] <- as_int(dt[[cn]])
    }
  }

  list(linear = dt_lin, nonlinear = dt_non)
}

# 共通 id での ARI を指標ごとに計算（kA/kBを併記）
compute_ari_table <- function(A, B, title = NULL) {
  if (is.null(A) || is.null(B)) {
    cat(sprintf("%s (no data)\n", ifelse(is.null(title), "[no title]", title)))
    return(invisible(NULL))
  }
  colsA <- setdiff(names(A), "id")
  colsB <- setdiff(names(B), "id")
  common <- intersect(colsA, colsB)
  if (length(common) == 0) {
    cat(sprintf("%s (no common columns)\n", ifelse(is.null(title), "[no title]", title)))
    return(invisible(NULL))
  }
  merged <- merge(A[, c("id", common), with = FALSE],
                  B[, c("id", common), with = FALSE],
                  by = "id", all = FALSE, suffixes = c(".A",".B"))
  if (nrow(merged) == 0) {
    cat(sprintf("%s (no overlapping ids)\n", ifelse(is.null(title), "[no title]", title)))
    return(invisible(NULL))
  }
  out <- data.table(condition = character(), n = integer(), kA = integer(), kB = integer(), ARI = numeric())
  for (cn in common) {
    x <- merged[[paste0(cn, ".A")]]
    y <- merged[[paste0(cn, ".B")]]
    ok <- !is.na(x) & !is.na(y)
    x <- x[ok]; y <- y[ok]
    n  <- length(x)
    kA <- if (n > 0) data.table::uniqueN(x) else 0L
    kB <- if (n > 0) data.table::uniqueN(y) else 0L
    ari <- if (n >= 2 && kA > 1 && kB > 1) adjustedRandIndex(x, y) else NA_real_
    out <- rbind(out, data.table(condition = cn, n = n, kA = kA, kB = kB, ARI = ari))
  }
  md  <- ifelse(grepl("^cumeig_", out$condition), 1L,
                ifelse(grepl("^top3_", out$condition), 2L, 3L))
  out <- out[order(md, condition)]
  cat(sprintf("\n%s\n", ifelse(is.null(title), "[Comparison]", title)))
  print(out, row.names = FALSE)
  invisible(out)
}

# 同一ステップ内の cumeig vs top3（各指標）の比較（kを併記）
compare_cumeig_vs_top3 <- function(DT, title) {
  if (is.null(DT)) { cat(sprintf("\n%s (no data)\n", title)); return(invisible(NULL)) }
  cols <- setdiff(names(DT), "id")
  idxs <- unique(sub("^(cumeig|top3)_", "", cols))
  idxs <- idxs[idxs != ""]
  out <- data.table(index = character(), n = integer(), k_cumeig = integer(), k_top3 = integer(), ARI = numeric())
  for (ix in idxs) {
    c1 <- paste0("cumeig_", ix)
    c2 <- paste0("top3_", ix)
    if (c1 %in% cols && c2 %in% cols) {
      tmp <- DT[, .(id, x = get(c1), y = get(c2))]
      tmp <- tmp[!is.na(x) & !is.na(y)]
      n  <- nrow(tmp)
      k1 <- if (n > 0) uniqueN(tmp$x) else 0L
      k2 <- if (n > 0) uniqueN(tmp$y) else 0L
      ari <- if (n >= 2 && k1 > 1 && k2 > 1) adjustedRandIndex(tmp$x, tmp$y) else NA_real_
      out <- rbind(out, data.table(index = ix, n = n, k_cumeig = k1, k_top3 = k2, ARI = ari))
    }
  }
  if (nrow(out) == 0) {
    cat(sprintf("\n%s (no cumeig/top3 pairs)\n", title))
  } else {
    cat(sprintf("\n%s\n", title))
    print(out[order(index)], row.names = FALSE)
  }
  invisible(out)
}

# --------- 実行 ---------
for (ds in datasets) {
  cat(sprintf("\n================ DATASET: %s ================\n", ds))

  f_new      <- latest_labels(file.path(paths$new,      ds))
  f_old      <- latest_labels(file.path(paths$olddata,  ds))
  f_signless <- latest_labels(file.path(paths$signless, ds))
  f_step2    <- latest_labels(file.path(paths$step2,    ds))

  cat("[FILES]\n")
  cat(sprintf("  new:      %s\n", ifelse(is.null(f_new), "NA", f_new)))
  cat(sprintf("  signless: %s\n", ifelse(is.null(f_signless), "NA", f_signless)))
  cat(sprintf("  olddata:  %s\n", ifelse(is.null(f_old), "NA", f_old)))
  if (!is.null(f_step2)) cat(sprintf("  step2:    %s\n", f_step2))

  NEW  <- if (!is.null(f_new)) read_labels_plain(f_new) else NULL
  OLD  <- if (!is.null(f_old)) read_labels_plain(f_old) else NULL
  SL   <- if (!is.null(f_signless)) read_labels_signless(f_signless) else NULL
  SL_L <- if (!is.null(SL)) SL$linear else NULL
  SL_N <- if (!is.null(SL)) SL$nonlinear else NULL
  STEP2<- if (!is.null(f_step2)) read_labels_plain(f_step2) else NULL

  compute_ari_table(NEW, SL_L, sprintf("-- NEW vs SIGNLESS(linear) [%s] --", ds))
  compute_ari_table(NEW, OLD,  sprintf("-- NEW vs OLDDATA [%s] --", ds))
  compute_ari_table(SL_L, SL_N, sprintf("-- SIGNLESS: linear vs nonlinear [%s] --", ds))

  compare_cumeig_vs_top3(NEW,  sprintf("-- Within NEW: cumeig vs top3 [%s] --", ds))
  compare_cumeig_vs_top3(OLD,  sprintf("-- Within OLDDATA: cumeig vs top3 [%s] --", ds))
  compare_cumeig_vs_top3(SL_L, sprintf("-- Within SIGNLESS(linear): cumeig vs top3 [%s] --", ds))
  compare_cumeig_vs_top3(SL_N, sprintf("-- Within SIGNLESS(nonlinear): cumeig vs top3 [%s] --", ds))
}

cat("\nDONE: All ARI comparisons printed to console.\n")



================ DATASET: OH ================
[FILES]
  new:      /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/OH/cluster_labels_OH_20250910_093155.csv
  signless: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3.2_signlessCorr_MDS_20250904/OH/cluster_labels_OH_20250904_160436.csv
  olddata:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3.3_olddata_MDS_20250904/OH/cluster_labels_OH_20250910_093156.csv

-- NEW vs SIGNLESS(linear) [OH] --
         condition     n    kA    kB        ARI
            <char> <int> <int> <int>      <num>
         cumeig_ch   394     2     2 0.72624273
         cumeig_db   394     3    96 0.02327565
       cumeig_dunn   394     4     4 0.77094799
        cumeig_gap   394     2     2 0.72624273
 cumeig_ptbiserial   394    15   100 0.09572465
 cumeig_silhouette   394     2   100 0.01917567
           top3_ch   394   100   100 0.24315860
           top3_db   394    97   100 0.24510104
    


DONE: All ARI comparisons printed to console.


In [20]:
# =============================================
# クラスター比較（ARI）スクリプト - k表示 + Base N 表示付き
# =============================================

suppressPackageStartupMessages({
  if (!requireNamespace("data.table", quietly = TRUE)) install.packages("data.table")
  if (!requireNamespace("mclust", quietly = TRUE)) install.packages("mclust")
  library(data.table)
  library(mclust)  # adjustedRandIndex
})

# ルートと step パス（必要に応じて調整）
root <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
paths <- list(
  step2    = file.path(root, "STEP2_old_MDS_20250904"),
  new      = file.path(root, "STEP3_new_MDS_20250904"),
  signless = file.path(root, "STEP3.2_signlessCorr_MDS_20250904"),
  olddata  = file.path(root, "STEP3.3_olddata_MDS_20250904")
)
datasets <- c("OH","FP")

# --------- ユーティリティ ---------
latest_labels <- function(dir_ds) {
  if (!dir.exists(dir_ds)) return(NULL)
  fs <- list.files(dir_ds, pattern = "^cluster_labels_.*\\.csv$", full.names = TRUE)
  if (length(fs) == 0) return(NULL)
  fs[which.max(file.mtime(fs))]
}

read_labels_plain <- function(f) {
  dt <- fread(f)
  if (!"id" %in% names(dt)) setnames(dt, 1, "id")
  dt[, id := as.character(id)]
  for (cn in setdiff(names(dt), "id")) {
    x <- dt[[cn]]
    if (!(is.integer(x) || is.numeric(x))) dt[[cn]] <- as.integer(factor(x))
  }
  dt
}

read_labels_signless <- function(f) {
  dt <- fread(f)
  if (!"id" %in% names(dt)) setnames(dt, 1, "id")
  dt[, id := as.character(id)]

  nms <- setdiff(names(dt), "id")
  lin <- nms[grepl("^linear_", nms)]
  non <- nms[grepl("^nonlinear_", nms)]
  oth <- setdiff(nms, c(lin, non))

  as_int <- function(v) if (is.integer(v) || is.numeric(v)) as.integer(v) else as.integer(factor(v))

  dt_lin <- if (length(lin) > 0) {
    tmp <- data.table(id = dt$id)
    for (cn in lin) tmp[[sub("^linear_", "", cn)]] <- as_int(dt[[cn]])
    tmp
  } else NULL

  dt_non <- if (length(non) > 0) {
    tmp <- data.table(id = dt$id)
    for (cn in non) tmp[[sub("^nonlinear_", "", cn)]] <- as_int(dt[[cn]])
    tmp
  } else NULL

  if (length(oth) > 0) {
    if (is.null(dt_lin)) dt_lin <- data.table(id = dt$id)
    for (cn in oth) {
      newcn <- sub("_linear$", "", cn)
      dt_lin[[newcn]] <- as_int(dt[[cn]])
    }
  }

  list(linear = dt_lin, nonlinear = dt_non)
}

fmt_int <- function(x) if (is.null(x) || is.na(x)) "NA" else as.character(x)

# 共通 id での ARI を指標ごとに計算（kA/kBを併記）
compute_ari_table <- function(A, B, title = NULL, A_name = "A", B_name = "B") {
  cat(sprintf("\n%s\n", ifelse(is.null(title), "[Comparison]", title)))

  nA <- if (!is.null(A)) nrow(A) else NA_integer_
  nB <- if (!is.null(B)) nrow(B) else NA_integer_
  ov <- if (!is.null(A) && !is.null(B)) length(intersect(A$id, B$id)) else NA_integer_
  cat(sprintf("  Base IDs: %s=%s, %s=%s, overlap=%s\n",
              A_name, fmt_int(nA), B_name, fmt_int(nB), fmt_int(ov)))

  if (is.null(A) || is.null(B)) { cat("  (no data)\n"); return(invisible(NULL)) }

  colsA <- setdiff(names(A), "id")
  colsB <- setdiff(names(B), "id")
  common <- intersect(colsA, colsB)
  if (length(common) == 0) { cat("  (no common columns)\n"); return(invisible(NULL)) }

  merged <- merge(A[, c("id", common), with = FALSE],
                  B[, c("id", common), with = FALSE],
                  by = "id", all = FALSE, suffixes = c(".A",".B"))
  if (nrow(merged) == 0) { cat("  (no overlapping ids)\n"); return(invisible(NULL)) }

  out <- data.table(condition = character(), n = integer(), kA = integer(), kB = integer(), ARI = numeric())
  for (cn in common) {
    x <- merged[[paste0(cn, ".A")]]
    y <- merged[[paste0(cn, ".B")]]
    ok <- !is.na(x) & !is.na(y)
    x <- x[ok]; y <- y[ok]
    n  <- length(x)
    kA <- if (n > 0) data.table::uniqueN(x) else 0L
    kB <- if (n > 0) data.table::uniqueN(y) else 0L
    ari <- if (n >= 2 && kA > 1 && kB > 1) adjustedRandIndex(x, y) else NA_real_
    out <- rbind(out, data.table(condition = cn, n = n, kA = kA, kB = kB, ARI = ari))
  }
  md  <- ifelse(grepl("^cumeig_", out$condition), 1L,
                ifelse(grepl("^top3_", out$condition), 2L, 3L))
  out <- out[order(md, condition)]
  print(out, row.names = FALSE)
  invisible(out)
}

# 同一ステップ内の cumeig vs top3（各指標）の比較（kを併記）
compare_cumeig_vs_top3 <- function(DT, title) {
  cat(sprintf("\n%s\n", title))
  if (is.null(DT)) { cat("  Base IDs: NA\n  (no data)\n"); return(invisible(NULL)) }
  cat(sprintf("  Base IDs: N=%s\n", fmt_int(nrow(DT))))

  cols <- setdiff(names(DT), "id")
  idxs <- unique(sub("^(cumeig|top3)_", "", cols))
  idxs <- idxs[idxs != ""]
  out <- data.table(index = character(), n = integer(), k_cumeig = integer(), k_top3 = integer(), ARI = numeric())
  for (ix in idxs) {
    c1 <- paste0("cumeig_", ix)
    c2 <- paste0("top3_", ix)
    if (c1 %in% cols && c2 %in% cols) {
      tmp <- DT[, .(id, x = get(c1), y = get(c2))]
      tmp <- tmp[!is.na(x) & !is.na(y)]
      n  <- nrow(tmp)
      k1 <- if (n > 0) uniqueN(tmp$x) else 0L
      k2 <- if (n > 0) uniqueN(tmp$y) else 0L
      ari <- if (n >= 2 && k1 > 1 && k2 > 1) adjustedRandIndex(tmp$x, tmp$y) else NA_real_
      out <- rbind(out, data.table(index = ix, n = n, k_cumeig = k1, k_top3 = k2, ARI = ari))
    }
  }
  if (nrow(out) == 0) {
    cat("  (no cumeig/top3 pairs)\n")
  } else {
    print(out[order(index)], row.names = FALSE)
  }
  invisible(out)
}

# --------- 実行 ---------
cat(sprintf("Resolved root: %s\n", root))
for (ds in datasets) {
  cat(sprintf("\n================ DATASET: %s ================\n", ds))

  f_new      <- latest_labels(file.path(paths$new,      ds))
  f_old      <- latest_labels(file.path(paths$olddata,  ds))
  f_signless <- latest_labels(file.path(paths$signless, ds))
  f_step2    <- latest_labels(file.path(paths$step2,    ds))

  cat("[FILES]\n")
  cat(sprintf("  new:      %s\n", ifelse(is.null(f_new), "NA", f_new)))
  cat(sprintf("  signless: %s\n", ifelse(is.null(f_signless), "NA", f_signless)))
  cat(sprintf("  olddata:  %s\n", ifelse(is.null(f_old), "NA", f_old)))
  if (!is.null(f_step2)) cat(sprintf("  step2:    %s\n", f_step2))

  NEW  <- if (!is.null(f_new)) read_labels_plain(f_new) else NULL
  OLD  <- if (!is.null(f_old)) read_labels_plain(f_old) else NULL
  SL   <- if (!is.null(f_signless)) read_labels_signless(f_signless) else NULL
  SL_L <- if (!is.null(SL)) SL$linear else NULL
  SL_N <- if (!is.null(SL)) SL$nonlinear else NULL
  STEP2<- if (!is.null(f_step2)) read_labels_plain(f_step2) else NULL

  compute_ari_table(NEW, SL_L, sprintf("-- NEW vs SIGNLESS(linear) [%s] --", ds),
                    A_name = "NEW", B_name = "SIGNLESS(L)")
  compute_ari_table(NEW, OLD,  sprintf("-- NEW vs OLDDATA [%s] --", ds),
                    A_name = "NEW", B_name = "OLDDATA")
  compute_ari_table(SL_L, SL_N, sprintf("-- SIGNLESS: linear vs nonlinear [%s] --", ds),
                    A_name = "SIGNLESS(L)", B_name = "SIGNLESS(NL)")

  compare_cumeig_vs_top3(NEW,  sprintf("-- Within NEW: cumeig vs top3 [%s] --", ds))
  compare_cumeig_vs_top3(OLD,  sprintf("-- Within OLDDATA: cumeig vs top3 [%s] --", ds))
  compare_cumeig_vs_top3(SL_L, sprintf("-- Within SIGNLESS(linear): cumeig vs top3 [%s] --", ds))
  compare_cumeig_vs_top3(SL_N, sprintf("-- Within SIGNLESS(nonlinear): cumeig vs top3 [%s] --", ds))
}

cat("\nDONE: All ARI comparisons printed to console.\n")


Resolved root: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No

================ DATASET: OH ================
[FILES]
  new:      /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/OH/cluster_labels_OH_20250910_093155.csv
  signless: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3.2_signlessCorr_MDS_20250904/OH/cluster_labels_OH_20250904_160436.csv
  olddata:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3.3_olddata_MDS_20250904/OH/cluster_labels_OH_20250910_093156.csv

-- NEW vs SIGNLESS(linear) [OH] --
  Base IDs: NEW=394, SIGNLESS(L)=394, overlap=394
         condition     n    kA    kB        ARI
            <char> <int> <int> <int>      <num>
         cumeig_ch   394     2     2 0.72624273
         cumeig_db   394     3    96 0.02327565
       cumeig_dunn   394     4     4 0.77094799
        cumeig_gap   394     2     2 0.72624273
 cumeig_ptbiserial   394    15   100 0.09572465
 cumeig_silhouette   39


DONE: All ARI comparisons printed to console.


In [30]:
# ============================================================
# 解析済みラベルCSVを用いた比較（全件出力版）
# - 対象ステップ：NEW, SIGNLESS(linear/nonlinear), OLDDATA
# - データセット：OH, FP
# - 指標：ARI, NMI, AMI, VI（VIは自前実装でフォールバック）
# - 出力：表ごとにCSV（専用の新規フォルダを作成）
# ============================================================

## ---- パッケージ ----
safeload <- function(pkg){
  if (!suppressWarnings(require(pkg, character.only = TRUE))) {
    install.packages(pkg, repos = "https://cloud.r-project.org")
    library(pkg, character.only = TRUE)
  }
}
safeload("data.table")
safeload("dplyr")
safeload("stringr")
safeload("mclust")
safeload("aricode")  # NMI/AMI用（VIは自前実装でフォールバック）

## ---- VI の安全実装＆ラッパー ----
vi_from_xy <- function(x, y){
  n <- length(x)
  if (n <= 1L) return(NA_real_)
  tab <- table(x, y)
  P   <- tab / n
  pi  <- rowSums(P)
  pj  <- colSums(P)
  H <- function(p){ p <- p[p > 0]; if (!length(p)) return(0); -sum(p * log(p)) }
  Hx <- H(pi); Hy <- H(pj)
  PxPy <- outer(pi, pj)
  idx  <- (P > 0) & (PxPy > 0)
  Ixy  <- if (!any(idx)) 0 else sum(P[idx] * (log(P[idx]) - log(PxPy[idx])))
  Hx + Hy - 2 * Ixy
}
safe_vi <- function(x, y){
  tryCatch({
    if (requireNamespace("aricode", quietly = TRUE)) {
      if ("VI" %in% getNamespaceExports("aricode")) {
        get("VI", envir = asNamespace("aricode"))(x, y)
      } else if (exists("VI", envir = asNamespace("aricode"), inherits = FALSE)) {
        get("VI", envir = asNamespace("aricode"))(x, y)
      } else {
        vi_from_xy(x, y)
      }
    } else vi_from_xy(x, y)
  }, error = function(e) vi_from_xy(x, y))
}
VI <- function(x, y) safe_vi(x, y)

## ---- 設定 ----
ROOT <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
STEP_DIR <- list(
  NEW      = file.path(ROOT, "STEP3_new_MDS_20250904"),
  SIGNLESS = file.path(ROOT, "STEP3.2_signlessCorr_MDS_20250904"),
  OLDDATA  = file.path(ROOT, "STEP3.3_olddata_MDS_20250904")
)
DATASETS <- c("OH", "FP")
TS <- format(Sys.time(), "%Y%m%d_%H%M%S")
OUT_DIR <- file.path(ROOT, paste0("COMPARE_EXPORTS_ALL_", TS))
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

## ---- ヘルパー：ファイル解決＆読込 ----
latest_labels_csv <- function(dir_step, dataset){
  d <- file.path(dir_step, dataset)
  fs <- list.files(d, pattern = "^cluster_labels_.*\\.csv$", full.names = TRUE)
  if (!length(fs)) return(NA_character_)
  info <- file.info(fs)
  fs[order(info$mtime, decreasing = TRUE)][1]
}
read_labels <- function(csv_path){
  if (is.na(csv_path)) return(NULL)
  dt <- data.table::fread(csv_path)
  if (!("id" %in% names(dt))) setnames(dt, names(dt)[1], "id")
  lab_cols <- setdiff(names(dt), "id")
  for (cn in lab_cols){
    if (is.character(dt[[cn]]) || is.factor(dt[[cn]])) {
      suppressWarnings(dt[[cn]] <- as.integer(as.character(dt[[cn]])))
    }
  }
  dt
}

## ---- アライン ----
align_by_id <- function(dtA, dtB){
  if (is.null(dtA) || is.null(dtB)) return(NULL)
  m <- merge(dtA, dtB, by = "id", suffixes = c(".A", ".B"))
  list(
    A = m[, c("id", grep("\\.A$", names(m), value = TRUE)), with = FALSE],
    B = m[, c("id", grep("\\.B$", names(m), value = TRUE)), with = FALSE],
    n_overlap = nrow(m)
  )
}

## ---- 指標計算（ペア） ----
pair_metrics <- function(a, b){
  ok <- which(!is.na(a) & !is.na(b))
  n  <- length(ok)
  kA <- if (n > 0) length(unique(a[ok])) else 0L
  kB <- if (n > 0) length(unique(b[ok])) else 0L
  if (n <= 1L || kA < 2L || kB < 2L){
    return(list(n=n, kA=kA, kB=kB,
                ARI=NA_real_, NMI=NA_real_, AMI=NA_real_, VI=NA_real_))
  }
  x <- as.vector(a[ok]); y <- as.vector(b[ok])
  list(
    n   = n, kA = kA, kB = kB,
    ARI = mclust::adjustedRandIndex(x, y),
    NMI = aricode::NMI(x, y),
    AMI = aricode::AMI(x, y),
    VI  = safe_vi(x, y)
  )
}

## ---- WITHIN-STEP 全ペア ----
within_step_allpairs <- function(step_name, lbl_dt){
  if (is.null(lbl_dt)) return(data.table())
  lab_cols <- setdiff(names(lbl_dt), "id")
  if (length(lab_cols) < 2) return(data.table())
  combs <- t(combn(lab_cols, 2))
  res <- rbindlist(lapply(seq_len(nrow(combs)), function(i){
    a <- combs[i, 1]; b <- combs[i, 2]
    m <- pair_metrics(lbl_dt[[a]], lbl_dt[[b]])
    data.table(block = sprintf("WITHIN:%s", step_name),
               from=a, to=b, n=m$n, kA=m$kA, kB=m$kB,
               ARI=m$ARI, NMI=m$NMI, AMI=m$AMI, VI=m$VI)
  }), fill = TRUE)
  res[]
}

## ---- CROSS-STEP 同名列比較 ----
cross_step_samecol <- function(nameA, dtA, nameB, dtB){
  if (is.null(dtA) || is.null(dtB)) return(data.table())
  labsA <- setdiff(names(dtA), "id")
  labsB <- setdiff(names(dtB), "id")
  common <- intersect(labsA, labsB)
  ab <- align_by_id(dtA, dtB)
  if (is.null(ab)) return(data.table())
  if (!length(common)) return(data.table())
  res <- rbindlist(lapply(common, function(cn){
    a <- ab$A[[paste0(cn, ".A")]]; b <- ab$B[[paste0(cn, ".B")]]
    m <- pair_metrics(a, b)
    data.table(block = sprintf("CROSS:%s_vs_%s", nameA, nameB),
               condition = cn, n = m$n, kA = m$kA, kB = m$kB,
               ARI = m$ARI, NMI = m$NMI, AMI = m$AMI, VI = m$VI)
  }), fill = TRUE)
  res[]
}

## ---- SIGNLESS(線形 vs 非線形) ----
signless_linear_vs_nonlinear <- function(lbl_dt_sign){
  if (is.null(lbl_dt_sign)) return(data.table())
  cols <- setdiff(names(lbl_dt_sign), "id")
  lin  <- cols[stringr::str_starts(cols, "linear_")]
  non  <- cols[stringr::str_starts(cols, "nonlinear_")]
  core_lin <- sub("^linear_", "", lin)
  core_non <- sub("^nonlinear_", "", non)
  common_core <- intersect(core_lin, core_non)
  if (!length(common_core)) return(data.table())
  res <- rbindlist(lapply(common_core, function(core){
    a <- paste0("linear_", core)
    b <- paste0("nonlinear_", core)
    m <- pair_metrics(lbl_dt_sign[[a]], lbl_dt_sign[[b]])
    data.table(block = "SIGNLESS:linear_vs_nonlinear",
               condition = core, n = m$n, kA = m$kA, kB = m$kB,
               ARI = m$ARI, NMI = m$NMI, AMI = m$AMI, VI = m$VI)
  }), fill = TRUE)
  res[]
}

## ---- 同ステップ内 cumeig vs top3 ----
within_cumeig_vs_top3 <- function(step_name, lbl_dt){
  if (is.null(lbl_dt)) return(data.table())
  cols <- setdiff(names(lbl_dt), "id")
  # 通常（NEW/OLDDATA）
  if (any(stringr::str_starts(cols, "cumeig_")) || any(stringr::str_starts(cols, "top3_"))){
    common_core <- intersect(sub("^cumeig_", "", cols[stringr::str_starts(cols, "cumeig_")]),
                             sub("^top3_",   "", cols[stringr::str_starts(cols, "top3_")]))
    if (!length(common_core)) return(data.table())
    return(rbindlist(lapply(common_core, function(core){
      a <- paste0("cumeig_", core); b <- paste0("top3_", core)
      m <- pair_metrics(lbl_dt[[a]], lbl_dt[[b]])
      data.table(block = sprintf("WITHIN:%s cumeig_vs_top3", step_name),
                 index = core, n = m$n, k_cumeig = m$kA, k_top3 = m$kB,
                 ARI = m$ARI, NMI = m$NMI, AMI = m$AMI, VI = m$VI)
    }), fill = TRUE))
  }
  # SIGNLESS（linear/非linear それぞれ）
  do_mode <- function(mode){
    cx <- paste0("^", mode, "_cumeig_"); tx <- paste0("^", mode, "_top3_")
    core <- intersect(sub(cx, "", cols[stringr::str_starts(cols, cx)]),
                      sub(tx, "", cols[stringr::str_starts(cols, tx)]))
    if (!length(core)) return(data.table())
    rbindlist(lapply(core, function(k){
      a <- paste0(mode, "_cumeig_", k); b <- paste0(mode, "_top3_", k)
      m <- pair_metrics(lbl_dt[[a]], lbl_dt[[b]])
      data.table(block = sprintf("WITHIN:SIGNLESS(%s) cumeig_vs_top3", mode),
                 index = k, n = m$n, k_cumeig = m$kA, k_top3 = m$kB,
                 ARI = m$ARI, NMI = m$NMI, AMI = m$AMI, VI = m$VI)
    }), fill = TRUE)
  }
  rbindlist(list(do_mode("linear"), do_mode("nonlinear")), fill = TRUE)
}

## ---- 書き出しヘルパー ----
write_tbl <- function(dt, path){
  dir.create(dirname(path), recursive = TRUE, showWarnings = FALSE)
  if (!nrow(dt)){
    message(sprintf("[SKIP empty] %s", path))
    return(invisible(FALSE))
  }
  data.table::fwrite(dt, path)
  message(sprintf("[WROTE] %s  (rows=%d, cols=%d)", path, nrow(dt), ncol(dt)))
  invisible(TRUE)
}

## ---- 実行 ----
cat(sprintf("Resolved root: %s\nOutput dir: %s\n\n", ROOT, OUT_DIR))

for (ds in DATASETS){
  cat(sprintf("================ DATASET: %s ================\n", ds))
  fp_new      <- latest_labels_csv(STEP_DIR$NEW, ds)
  fp_signless <- latest_labels_csv(STEP_DIR$SIGNLESS, ds)
  fp_old      <- latest_labels_csv(STEP_DIR$OLDDATA, ds)

  cat("[FILES]\n")
  cat(sprintf("  NEW     : %s \n", ifelse(is.na(fp_new), "NA", fp_new)))
  cat(sprintf("  SIGNLESS: %s \n", ifelse(is.na(fp_signless), "NA", fp_signless)))
  cat(sprintf("  OLDDATA : %s \n\n", ifelse(is.na(fp_old), "NA", fp_old)))

  dt_new      <- read_labels(fp_new)
  dt_signless <- read_labels(fp_signless)
  dt_old      <- read_labels(fp_old)

  ds_dir <- file.path(OUT_DIR, ds)
  dir.create(ds_dir, showWarnings = FALSE)

  # 1) WITHIN-STEP
  w_new <- within_step_allpairs("NEW", dt_new)
  write_tbl(w_new, file.path(ds_dir, "within_NEW_allpairs.csv"))

  w_sig <- within_step_allpairs("SIGNLESS", dt_signless)
  write_tbl(w_sig, file.path(ds_dir, "within_SIGNLESS_allpairs.csv"))

  w_old <- within_step_allpairs("OLDDATA", dt_old)
  write_tbl(w_old, file.path(ds_dir, "within_OLDDATA_allpairs.csv"))

  # 2) CROSS-STEP
  cs_ns <- cross_step_samecol("NEW", dt_new, "SIGNLESS", dt_signless)
  write_tbl(cs_ns, file.path(ds_dir, "cross_NEW_vs_SIGNLESS_samecol.csv"))

  cs_no <- cross_step_samecol("NEW", dt_new, "OLDDATA", dt_old)
  write_tbl(cs_no, file.path(ds_dir, "cross_NEW_vs_OLDDATA_samecol.csv"))

  cs_so <- cross_step_samecol("SIGNLESS", dt_signless, "OLDDATA", dt_old)
  write_tbl(cs_so, file.path(ds_dir, "cross_SIGNLESS_vs_OLDDATA_samecol.csv"))

  # 3) SIGNLESS: linear vs nonlinear
  sln <- signless_linear_vs_nonlinear(dt_signless)
  write_tbl(sln, file.path(ds_dir, "signless_linear_vs_nonlinear.csv"))

  # 4) cumeig vs top3
  wc_new <- within_cumeig_vs_top3("NEW", dt_new)
  write_tbl(wc_new, file.path(ds_dir, "within_NEW_cumeig_vs_top3.csv"))

  wc_old <- within_cumeig_vs_top3("OLDDATA", dt_old)
  write_tbl(wc_old, file.path(ds_dir, "within_OLDDATA_cumeig_vs_top3.csv"))

  wc_sig <- within_cumeig_vs_top3("SIGNLESS", dt_signless)
  write_tbl(wc_sig, file.path(ds_dir, "within_SIGNLESS_cumeig_vs_top3.csv"))

  cat("\n")
}

cat(sprintf("DONE. CSVはここにあります: %s\n", OUT_DIR))


Resolved root: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No
Output dir: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_ALL_20250911_120716

================ DATASET: OH ================
[FILES]
  NEW     : /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/OH/cluster_labels_OH_20250910_093155.csv 
  SIGNLESS: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3.2_signlessCorr_MDS_20250904/OH/cluster_labels_OH_20250904_160436.csv 
  OLDDATA : /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3.3_olddata_MDS_20250904/OH/cluster_labels_OH_20250910_093156.csv 



[WROTE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_ALL_20250911_120716/OH/within_NEW_allpairs.csv  (rows=66, cols=10)

[WROTE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_ALL_20250911_120716/OH/within_SIGNLESS_allpairs.csv  (rows=276, cols=10)

[WROTE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_ALL_20250911_120716/OH/within_OLDDATA_allpairs.csv  (rows=66, cols=10)

[SKIP empty] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_ALL_20250911_120716/OH/cross_NEW_vs_SIGNLESS_samecol.csv

[WROTE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_ALL_20250911_120716/OH/cross_NEW_vs_OLDDATA_samecol.csv  (rows=12, cols=9)

[SKIP empty] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_ALL_20250911_120716/OH/cross_SIGNLESS_vs_OLDDATA_samecol.csv

[WROTE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_


================ DATASET: FP ================
[FILES]
  NEW     : /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3_new_MDS_20250904/FP/cluster_labels_FP_20250910_093156.csv 
  SIGNLESS: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3.2_signlessCorr_MDS_20250904/FP/cluster_labels_FP_20250904_180435.csv 
  OLDDATA : /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3.3_olddata_MDS_20250904/FP/cluster_labels_FP_20250910_093156.csv 



[WROTE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_ALL_20250911_120716/FP/within_NEW_allpairs.csv  (rows=66, cols=10)

[WROTE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_ALL_20250911_120716/FP/within_SIGNLESS_allpairs.csv  (rows=276, cols=10)

[WROTE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_ALL_20250911_120716/FP/within_OLDDATA_allpairs.csv  (rows=66, cols=10)

[SKIP empty] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_ALL_20250911_120716/FP/cross_NEW_vs_SIGNLESS_samecol.csv

[WROTE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_ALL_20250911_120716/FP/cross_NEW_vs_OLDDATA_samecol.csv  (rows=12, cols=9)

[SKIP empty] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_ALL_20250911_120716/FP/cross_SIGNLESS_vs_OLDDATA_samecol.csv

[WROTE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_


DONE. CSV<U+306F><U+3053><U+3053><U+306B><U+3042><U+308A><U+307E><U+3059>: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_ALL_20250911_120716


In [31]:
# ============================================================
# 解析済みラベルCSVを用いた比較（k<=30 フィルタ版）
# - 片方でも k ≤ 30 の行だけを各表から抽出してCSV保存
# - 出力は専用の新規フォルダに表ごとCSV
# ============================================================

## ---- パッケージ & VIラッパ（上と同じ） ----
safeload <- function(pkg){
  if (!suppressWarnings(require(pkg, character.only = TRUE))) {
    install.packages(pkg, repos = "https://cloud.r-project.org")
    library(pkg, character.only = TRUE)
  }
}
safeload("data.table"); safeload("dplyr"); safeload("stringr")
safeload("mclust");     safeload("aricode")

vi_from_xy <- function(x, y){
  n <- length(x); if (n <= 1L) return(NA_real_)
  tab <- table(x, y); P <- tab / n; pi <- rowSums(P); pj <- colSums(P)
  H <- function(p){ p <- p[p > 0]; if (!length(p)) return(0); -sum(p * log(p)) }
  Hx <- H(pi); Hy <- H(pj); PxPy <- outer(pi, pj); idx <- (P > 0) & (PxPy > 0)
  Ixy  <- if (!any(idx)) 0 else sum(P[idx] * (log(P[idx]) - log(PxPy[idx])))
  Hx + Hy - 2 * Ixy
}
safe_vi <- function(x, y){
  tryCatch({
    if (requireNamespace("aricode", quietly = TRUE)) {
      if ("VI" %in% getNamespaceExports("aricode")) {
        get("VI", envir = asNamespace("aricode"))(x, y)
      } else if (exists("VI", envir = asNamespace("aricode"), inherits = FALSE)) {
        get("VI", envir = asNamespace("aricode"))(x, y)
      } else vi_from_xy(x, y)
    } else vi_from_xy(x, y)
  }, error = function(e) vi_from_xy(x, y))
}
VI <- function(x, y) safe_vi(x, y)

## ---- 設定 ----
FILTER_K <- 30  # 片方でも k ≤ 30 を残す
ROOT <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
STEP_DIR <- list(
  NEW      = file.path(ROOT, "STEP3_new_MDS_20250904"),
  SIGNLESS = file.path(ROOT, "STEP3.2_signlessCorr_MDS_20250904"),
  OLDDATA  = file.path(ROOT, "STEP3.3_olddata_MDS_20250904")
)
DATASETS <- c("OH", "FP")
TS <- format(Sys.time(), "%Y%m%d_%H%M%S")
OUT_DIR <- file.path(ROOT, paste0("COMPARE_EXPORTS_KLE30_", TS))
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

## ---- ファイル解決・読込（上と同じ） ----
latest_labels_csv <- function(dir_step, dataset){
  d <- file.path(dir_step, dataset)
  fs <- list.files(d, pattern = "^cluster_labels_.*\\.csv$", full.names = TRUE)
  if (!length(fs)) return(NA_character_)
  info <- file.info(fs)
  fs[order(info$mtime, decreasing = TRUE)][1]
}
read_labels <- function(csv_path){
  if (is.na(csv_path)) return(NULL)
  dt <- data.table::fread(csv_path)
  if (!("id" %in% names(dt))) setnames(dt, names(dt)[1], "id")
  lab_cols <- setdiff(names(dt), "id")
  for (cn in lab_cols){
    if (is.character(dt[[cn]]) || is.factor(dt[[cn]])) {
      suppressWarnings(dt[[cn]] <- as.integer(as.character(dt[[cn]])))
    }
  }
  dt
}
align_by_id <- function(dtA, dtB){
  if (is.null(dtA) || is.null(dtB)) return(NULL)
  m <- merge(dtA, dtB, by = "id", suffixes = c(".A", ".B"))
  list(
    A = m[, c("id", grep("\\.A$", names(m), value = TRUE)), with = FALSE],
    B = m[, c("id", grep("\\.B$", names(m), value = TRUE)), with = FALSE],
    n_overlap = nrow(m)
  )
}

## ---- 指標計算（ペア） ----
pair_metrics <- function(a, b){
  ok <- which(!is.na(a) & !is.na(b))
  n  <- length(ok)
  kA <- if (n > 0) length(unique(a[ok])) else 0L
  kB <- if (n > 0) length(unique(b[ok])) else 0L
  if (n <= 1L || kA < 2L || kB < 2L){
    return(list(n=n, kA=kA, kB=kB,
                ARI=NA_real_, NMI=NA_real_, AMI=NA_real_, VI=NA_real_))
  }
  x <- as.vector(a[ok]); y <- as.vector(b[ok])
  list(
    n   = n, kA = kA, kB = kB,
    ARI = mclust::adjustedRandIndex(x, y),
    NMI = aricode::NMI(x, y),
    AMI = aricode::AMI(x, y),
    VI  = safe_vi(x, y)
  )
}

## ---- 各ブロック生成（上と同じ） ----
within_step_allpairs <- function(step_name, lbl_dt){
  if (is.null(lbl_dt)) return(data.table())
  lab_cols <- setdiff(names(lbl_dt), "id")
  if (length(lab_cols) < 2) return(data.table())
  combs <- t(combn(lab_cols, 2))
  res <- rbindlist(lapply(seq_len(nrow(combs)), function(i){
    a <- combs[i, 1]; b <- combs[i, 2]
    m <- pair_metrics(lbl_dt[[a]], lbl_dt[[b]])
    data.table(block = sprintf("WITHIN:%s", step_name),
               from=a, to=b, n=m$n, kA=m$kA, kB=m$kB,
               ARI=m$ARI, NMI=m$NMI, AMI=m$AMI, VI=m$VI)
  }), fill = TRUE)
  res[]
}
cross_step_samecol <- function(nameA, dtA, nameB, dtB){
  if (is.null(dtA) || is.null(dtB)) return(data.table())
  labsA <- setdiff(names(dtA), "id")
  labsB <- setdiff(names(dtB), "id")
  common <- intersect(labsA, labsB)
  ab <- align_by_id(dtA, dtB)
  if (is.null(ab)) return(data.table())
  if (!length(common)) return(data.table())
  res <- rbindlist(lapply(common, function(cn){
    a <- ab$A[[paste0(cn, ".A")]]; b <- ab$B[[paste0(cn, ".B")]]
    m <- pair_metrics(a, b)
    data.table(block = sprintf("CROSS:%s_vs_%s", nameA, nameB),
               condition = cn, n = m$n, kA = m$kA, kB = m$kB,
               ARI = m$ARI, NMI = m$NMI, AMI = m$AMI, VI = m$VI)
  }), fill = TRUE)
  res[]
}
signless_linear_vs_nonlinear <- function(lbl_dt_sign){
  if (is.null(lbl_dt_sign)) return(data.table())
  cols <- setdiff(names(lbl_dt_sign), "id")
  lin  <- cols[stringr::str_starts(cols, "linear_")]
  non  <- cols[stringr::str_starts(cols, "nonlinear_")]
  core_lin <- sub("^linear_", "", lin)
  core_non <- sub("^nonlinear_", "", non)
  common_core <- intersect(core_lin, core_non)
  if (!length(common_core)) return(data.table())
  res <- rbindlist(lapply(common_core, function(core){
    a <- paste0("linear_", core)
    b <- paste0("nonlinear_", core)
    m <- pair_metrics(lbl_dt_sign[[a]], lbl_dt_sign[[b]])
    data.table(block = "SIGNLESS:linear_vs_nonlinear",
               condition = core, n = m$n, kA = m$kA, kB = m$kB,
               ARI = m$ARI, NMI = m$NMI, AMI = m$AMI, VI = m$VI)
  }), fill = TRUE)
  res[]
}
within_cumeig_vs_top3 <- function(step_name, lbl_dt){
  if (is.null(lbl_dt)) return(data.table())
  cols <- setdiff(names(lbl_dt), "id")
  if (any(stringr::str_starts(cols, "cumeig_")) || any(stringr::str_starts(cols, "top3_"))){
    common_core <- intersect(sub("^cumeig_", "", cols[stringr::str_starts(cols, "cumeig_")]),
                             sub("^top3_",   "", cols[stringr::str_starts(cols, "top3_")]))
    if (!length(common_core)) return(data.table())
    return(rbindlist(lapply(common_core, function(core){
      a <- paste0("cumeig_", core); b <- paste0("top3_", core)
      m <- pair_metrics(lbl_dt[[a]], lbl_dt[[b]])
      data.table(block = sprintf("WITHIN:%s cumeig_vs_top3", step_name),
                 index = core, n = m$n, k_cumeig = m$kA, k_top3 = m$kB,
                 ARI = m$ARI, NMI = m$NMI, AMI = m$AMI, VI = m$VI)
    }), fill = TRUE))
  }
  do_mode <- function(mode){
    cx <- paste0("^", mode, "_cumeig_"); tx <- paste0("^", mode, "_top3_")
    core <- intersect(sub(cx, "", cols[stringr::str_starts(cols, cx)]),
                      sub(tx, "", cols[stringr::str_starts(cols, tx)]))
    if (!length(core)) return(data.table())
    rbindlist(lapply(core, function(k){
      a <- paste0(mode, "_cumeig_", k); b <- paste0(mode, "_top3_", k)
      m <- pair_metrics(lbl_dt[[a]], lbl_dt[[b]])
      data.table(block = sprintf("WITHIN:SIGNLESS(%s) cumeig_vs_top3", mode),
                 index = k, n = m$n, k_cumeig = m$kA, k_top3 = m$kB,
                 ARI = m$ARI, NMI = m$NMI, AMI = m$AMI, VI = m$VI)
    }), fill = TRUE)
  }
  rbindlist(list(do_mode("linear"), do_mode("nonlinear")), fill = TRUE)
}

## ---- フィルタ & 書き出し ----
filter_k_pair <- function(dt){
  if (!nrow(dt)) return(dt)
  dt[(is.na(kA) | kA <= FILTER_K) | (is.na(kB) | kB <= FILTER_K)]
}
filter_k_cumeig_top3 <- function(dt){
  if (!nrow(dt)) return(dt)
  dt[(is.na(k_cumeig) | k_cumeig <= FILTER_K) | (is.na(k_top3) | k_top3 <= FILTER_K)]
}
write_tbl <- function(dt, path){
  dir.create(dirname(path), recursive = TRUE, showWarnings = FALSE)
  if (!nrow(dt)){
    message(sprintf("[SKIP empty] %s", path))
    return(invisible(FALSE))
  }
  data.table::fwrite(dt, path)
  message(sprintf("[WROTE] %s  (rows=%d, cols=%d)", path, nrow(dt), ncol(dt)))
  invisible(TRUE)
}

## ---- 実行 ----
cat(sprintf("Resolved root: %s\nOutput dir: %s\n\n", ROOT, OUT_DIR))

for (ds in DATASETS){
  cat(sprintf("================ DATASET: %s ================\n", ds))
  fp_new      <- latest_labels_csv(STEP_DIR$NEW, ds)
  fp_signless <- latest_labels_csv(STEP_DIR$SIGNLESS, ds)
  fp_old      <- latest_labels_csv(STEP_DIR$OLDDATA, ds)

  dt_new      <- read_labels(fp_new)
  dt_signless <- read_labels(fp_signless)
  dt_old      <- read_labels(fp_old)

  ds_dir <- file.path(OUT_DIR, ds)
  dir.create(ds_dir, showWarnings = FALSE)

  # 1) WITHIN-STEP（フィルタ）
  w_new <- filter_k_pair(within_step_allpairs("NEW", dt_new))
  write_tbl(w_new, file.path(ds_dir, "within_NEW_allpairs_kLE30.csv"))

  w_sig <- filter_k_pair(within_step_allpairs("SIGNLESS", dt_signless))
  write_tbl(w_sig, file.path(ds_dir, "within_SIGNLESS_allpairs_kLE30.csv"))

  w_old <- filter_k_pair(within_step_allpairs("OLDDATA", dt_old))
  write_tbl(w_old, file.path(ds_dir, "within_OLDDATA_allpairs_kLE30.csv"))

  # 2) CROSS-STEP（フィルタ）
  cs_ns <- filter_k_pair(cross_step_samecol("NEW", dt_new, "SIGNLESS", dt_signless))
  write_tbl(cs_ns, file.path(ds_dir, "cross_NEW_vs_SIGNLESS_samecol_kLE30.csv"))

  cs_no <- filter_k_pair(cross_step_samecol("NEW", dt_new, "OLDDATA", dt_old))
  write_tbl(cs_no, file.path(ds_dir, "cross_NEW_vs_OLDDATA_samecol_kLE30.csv"))

  cs_so <- filter_k_pair(cross_step_samecol("SIGNLESS", dt_signless, "OLDDATA", dt_old))
  write_tbl(cs_so, file.path(ds_dir, "cross_SIGNLESS_vs_OLDDATA_samecol_kLE30.csv"))

  # 3) SIGNLESS: linear vs nonlinear（フィルタ）
  sln <- filter_k_pair(signless_linear_vs_nonlinear(dt_signless))
  write_tbl(sln, file.path(ds_dir, "signless_linear_vs_nonlinear_kLE30.csv"))

  # 4) cumeig vs top3（フィルタ）
  wc_new <- filter_k_cumeig_top3(within_cumeig_vs_top3("NEW", dt_new))
  write_tbl(wc_new, file.path(ds_dir, "within_NEW_cumeig_vs_top3_kLE30.csv"))

  wc_old <- filter_k_cumeig_top3(within_cumeig_vs_top3("OLDDATA", dt_old))
  write_tbl(wc_old, file.path(ds_dir, "within_OLDDATA_cumeig_vs_top3_kLE30.csv"))

  wc_sig <- filter_k_cumeig_top3(within_cumeig_vs_top3("SIGNLESS", dt_signless))
  write_tbl(wc_sig, file.path(ds_dir, "within_SIGNLESS_cumeig_vs_top3_kLE30.csv"))

  cat("\n")
}

cat(sprintf("DONE. CSVはここにあります: %s\n", OUT_DIR))


Resolved root: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No
Output dir: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_KLE30_20250911_120733

================ DATASET: OH ================


[WROTE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_KLE30_20250911_120733/OH/within_NEW_allpairs_kLE30.csv  (rows=65, cols=10)

[WROTE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_KLE30_20250911_120733/OH/within_SIGNLESS_allpairs_kLE30.csv  (rows=240, cols=10)

[WROTE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_KLE30_20250911_120733/OH/within_OLDDATA_allpairs_kLE30.csv  (rows=65, cols=10)

[SKIP empty] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_KLE30_20250911_120733/OH/cross_NEW_vs_SIGNLESS_samecol_kLE30.csv

[WROTE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_KLE30_20250911_120733/OH/cross_NEW_vs_OLDDATA_samecol_kLE30.csv  (rows=10, cols=9)

[SKIP empty] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_KLE30_20250911_120733/OH/cross_SIGNLESS_vs_OLDDATA_samecol_kLE30.csv

[WROTE] /Volumes/csbdeep11/_


================ DATASET: FP ================


[WROTE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_KLE30_20250911_120733/FP/within_NEW_allpairs_kLE30.csv  (rows=38, cols=10)

[WROTE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_KLE30_20250911_120733/FP/within_SIGNLESS_allpairs_kLE30.csv  (rows=185, cols=10)

[WROTE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_KLE30_20250911_120733/FP/within_OLDDATA_allpairs_kLE30.csv  (rows=60, cols=10)

[SKIP empty] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_KLE30_20250911_120733/FP/cross_NEW_vs_SIGNLESS_samecol_kLE30.csv

[WROTE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_KLE30_20250911_120733/FP/cross_NEW_vs_OLDDATA_samecol_kLE30.csv  (rows=8, cols=9)

[SKIP empty] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_KLE30_20250911_120733/FP/cross_SIGNLESS_vs_OLDDATA_samecol_kLE30.csv

[WROTE] /Volumes/csbdeep11/_y


DONE. CSV<U+306F><U+3053><U+3053><U+306B><U+3042><U+308A><U+307E><U+3059>: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/COMPARE_EXPORTS_KLE30_20250911_120733
